# Multi-MiniPAR + LR1-B capture, with randomised acquisition settings

Extends `multi_minipar_capture.ipynb` with two things:

1. **The LR1-B reference spectrometer** joins the TIA and the MiniPARs. It is auto-gained to
   the ambient level, then read in lockstep with everything else.
2. **Randomised `gain` / `aint` / `astep` per MiniPAR**, validated against *both* saturation
   mechanisms before a capture starts — so the dataset spans the settings space without
   silently collecting clipped rows.

### What "far from saturation" means here

The two mechanisms are physically different and the firmware now reports them separately
(`SpectrometerResult.sat_flags`):

| mechanism | what happens | how it is checked |
|---|---|---|
| **digital** | a channel's ADC counter tops out | counts vs `full_scale = min(65535, (aint+1)·(astep+1))`, which *moves with exposure* — a channel can top out far below 65535 |
| **analog** | the photodiode/integrator front end clips | the hardware `ASAT` bit. **Not derivable from the counts** — it can fire while every channel still reads mid-scale |

Because analog saturation is a bare flag with no "distance" to read off, margin is measured by
*probing*: once a candidate comes back clean, **`ASTEP` is scaled by `ANALOG_MARGIN`** and the
flag re-read. If it trips there, the setting is too close to the edge and is redrawn. Set
`ANALOG_MARGIN = None` to skip the probe.

The probe moves `ASTEP` rather than `ATIME` on purpose. `ASTEP` is the *per-step* integration
time — what the integrator charges over, and therefore what makes the analog front end rail.
`ATIME` only sets how many completed steps are accumulated digitally, so raising it stresses
the counter, not the analog path, and would say nothing about analog headroom.

> **Firmware requirement.** This needs the build that reports saturation — `spec_sat` must
> answer `0x0000,none,full_scale,65535`. Section 2 checks for it and stops if it is missing.
> `spec_raw` is unchanged, so older captures stay comparable.

### Connections

MiniPARs and the TIA are USB CDC serial. The **LR1-B is USB HID**, not a COM port — it is
driven through [`lr1b.py`](lr1b.py) and needs `pip install hidapi`. It is optional: if it is
not plugged in the notebook runs without it.

## 1 · Imports and configuration

In [1]:
import json
import math
import random
import re
import time
from collections import deque
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import serial
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from IPython.display import clear_output, display

import sys
sys.path.insert(0, str(Path.cwd()))
# helpers.py stays in ../Scripts: it is shared with the pre-existing notebooks there.
sys.path.insert(0, '../Scripts')
from helpers import serial_ports

try:
    import lr1b
    from lr1b import LR1B, autogain
    LR1B_IMPORT_ERROR = None
except Exception as exc:                      # hidapi missing, or lr1b.py not alongside
    LR1B, autogain, LR1B_IMPORT_ERROR = None, None, exc

BAUD_RATE = 115200

# --- Device identity (mirrors Firmware/include/app/device_config.h) -------
PRODUCT_NAME = 'MiniPAR'
MAC_QUERY    = '[{"set":[{"label":"hello"}]}]'

# --- CSV schema ------------------------------------------------------------
SPECTRAL_CHANNELS = ['f1_415', 'f2_445', 'f3_480', 'f4_515',
                     'f5_555', 'f6_590', 'f7_630', 'f8_680',
                     'clear',  'nir']
SETTING_COLUMNS = ['gain', 'aint', 'astep']
# Saturation + LR1-B columns are appended after the original schema, so the columns
# multi_par_spec_regression.ipynb reads keep their names and meaning.
SAT_COLUMNS  = ['sat_kind', 'sat_mask', 'full_scale', 'headroom']
LR1B_COLUMNS = ['lr1b_exposure_ms', 'lr1b_peak', 'lr1b_total', 'lr1b_peak_nm',
                'lr1b_spectrum_file']
CSV_COLUMNS = (['timestamp', 'measurement', 'tia_par', 'miniPAR', 'metadata']
               + SPECTRAL_CHANNELS + SETTING_COLUMNS + SAT_COLUMNS + LR1B_COLUMNS)
N_SPEC = len(SPECTRAL_CHANNELS)

# --- Where readings are stored --------------------------------------------
_here = Path.cwd()
DATA_DIR = _here.parent / 'data' if (_here.parent / 'data').is_dir() else _here / 'data'
CSV_PATH = DATA_DIR / 'multi_par_spec_lr1b.csv'      # new file: the schema is wider
LR1B_DIR = DATA_DIR / 'lr1b_spectra'                 # one full spectrum per capture

# --- Stability criterion ---------------------------------------------------
STABLE_WINDOW   = 4
STABLE_TOL      = 0.01
STABLE_MAX_WAIT = 120.0     # s; the LR1-B adds its exposure to every sample
SETTLE_S        = 0.10
TIA_ABS_FLOOR   = 0.05
SPEC_ABS_FLOOR  = 8.0
LR1B_ABS_FLOOR  = 50.0      # LR1-B total, counts

# --- Randomised acquisition settings ---------------------------------------
RANDOMISE_SETTINGS = True
GAIN_CHOICES       = list(range(0, 11))     # register 0..10 -> 0.5x .. 512x
INT_TIME_MS_RANGE  = (10.0, 500.0)          # sampled log-uniformly
ASTEP_RANGE        = (99, 20000)
SAT_HEADROOM       = 0.85    # peak count must stay below this fraction of full scale
MIN_PEAK_COUNTS    = 200     # absolute floor: a setting must put the peak above this
MIN_SIGNAL_FRAC    = 0.00    # optional extra floor, as a fraction of full scale
RESPONSIVITY_GUIDED = True   # measure the device's response once, then draw only settings
                             # predicted to land the peak inside PEAK_BAND
PEAK_BAND          = (0.05, 0.70)   # predicted peak, as a fraction of full scale
PROBE_MIN_COUNTS   = 100     # the responsivity probe needs at least this many counts
PROBE_ASTEP        = 999     # per-step time for the probe (2.78 ms - safe for the analog path)
ANALOG_MARGIN      = 1.5     # probe factor for analog headroom; None disables the probe
MAX_SETTING_TRIES  = 12
RANDOM_SEED        = None    # set an int for reproducible draws

# --- LR1-B ------------------------------------------------------------------
LR1B_TARGET_COUNTS = 20_000
LR1B_FULL_SCALE    = 65535
LR1B_EXPOSURE_MIN  = 0.01
LR1B_EXPOSURE_MAX  = 2000.0
LR1B_GATES_STABILITY = False  # the LR1-B is read, but never holds up a capture
LR1B_READ_EVERY_SAMPLE = True # False = read it once after the others settle (faster when
                              # its exposure is long, but no longer simultaneous)

AS7341_TICK_S = 2.78e-6      # ADC step period

RNG = random.Random(RANDOM_SEED)

print(f'CSV target : {CSV_PATH}')
print(f'LR1-B dir  : {LR1B_DIR}')
print(f'Stability  : {STABLE_WINDOW} samples within {STABLE_TOL * 100:.1f} % on the TIA and '
      f'every MiniPAR' + ('' if LR1B_GATES_STABILITY else ' (LR1-B excluded)'))
print(f'Settings   : peak must exceed {MIN_PEAK_COUNTS} counts and stay under '
      f'{SAT_HEADROOM:.0%} of full scale')
if LR1B is None:
    print(f'LR1-B      : unavailable ({LR1B_IMPORT_ERROR}) - capture will run without it')

CSV target : c:\Users\LudovicoCaracciolo\.traycer\worktrees\jan-ingenhousz-institute__minipar\traycer-minipar-cosmic-elk\Scripts\data\multi_par_spec_lr1b.csv
LR1-B dir  : c:\Users\LudovicoCaracciolo\.traycer\worktrees\jan-ingenhousz-institute__minipar\traycer-minipar-cosmic-elk\Scripts\data\lr1b_spectra
Stability  : 4 samples within 1.0 % on the TIA and every MiniPAR (LR1-B excluded)
Settings   : peak must exceed 200 counts and stay under 85% of full scale


## 2 · Acquisition maths

`full_scale` is *not* a constant. AN000633 p.7 note 1: the integration time sets the full-scale
range, so at short exposures a channel tops out well below 65535. `full_scale_counts` mirrors
the firmware `spectrometerGetFullScale` exactly and is cross-checked against the device in
section 5.

`basic_count_divisor` is the same formula as the firmware `spectrometerGetBasicCountDivisor`
but keeps t_int in **seconds**, because everything downstream here (G·T ladders, dark-offset
fits) is expressed in gain-seconds. Firmware >= 1.06 uses **ms**, so its `spec` basic counts
are 1000x smaller than this one returns. Harmless as long as they are not mixed: this notebook
only ever reads `spec_raw` (raw ADC counts) and normalises host-side.

In [2]:
def gain_multiplier(gain_reg):
    """AS7341/AS7343 GAIN register -> multiplier: 0 -> 0.5x, n -> 2**(n-1)."""
    gain_reg = int(gain_reg)
    return 0.5 if gain_reg == 0 else float(1 << (gain_reg - 1))


def integration_time_s(aint, astep):
    return (int(aint) + 1) * (int(astep) + 1) * AS7341_TICK_S


def full_scale_counts(aint, astep):
    """ADC full scale for this exposure, capped at the 16-bit counter."""
    return min(0xFFFF, (int(aint) + 1) * (int(astep) + 1))


def basic_count_divisor(gain, aint, astep):
    """raw / this = basic count (gain- and exposure-normalised)."""
    return gain_multiplier(gain) * integration_time_s(aint, astep)


def aint_for(target_time_s, astep):
    """ATIME that gets closest to `target_time_s` at this ASTEP, or None if out of range."""
    aint = round(target_time_s / ((int(astep) + 1) * AS7341_TICK_S)) - 1
    return int(aint) if 0 <= aint <= 255 else None


def describe_settings(gain, aint, astep):
    return (f'gain={gain} ({gain_multiplier(gain):g}x)  aint={aint}  astep={astep}  '
            f'tint={integration_time_s(aint, astep) * 1e3:.1f} ms  '
            f'full_scale={full_scale_counts(aint, astep)}')


print(describe_settings(2, 100, 999), '  <- firmware defaults')
print(describe_settings(5, 10, 999), '  <- note the reduced full scale')

gain=2 (2x)  aint=100  astep=999  tint=280.8 ms  full_scale=65535   <- firmware defaults
gain=5 (16x)  aint=10  astep=999  tint=30.6 ms  full_scale=11000   <- note the reduced full scale


## 3 · Find the instruments

MiniPARs and the TIA are probed exactly as in `multi_minipar_capture.ipynb` (CRLF-terminated,
DTR/RTS held low for the ESP32). The LR1-B is enumerated over HID and never appears as a COM
port, so it cannot collide with the serial scan.

In [3]:
def _read_until_quiet(ser, quiet_s=0.35, max_s=2.5):
    """Read everything the port sends until it has been silent for `quiet_s`."""
    chunks, deadline, last = [], time.monotonic() + max_s, time.monotonic()
    while time.monotonic() < deadline:
        data = ser.read_all()
        if data:
            chunks.append(data)
            last = time.monotonic()
        elif time.monotonic() - last >= quiet_s:
            break
        time.sleep(0.05)
    return b''.join(chunks).decode(errors='ignore')


def probe_minipar(port, timeout=2.0):
    """Firmware banner if `port` is a MiniPAR, else None.

    CRLF, not bare LF: the MiniPAR firmware drops '\r', but MicroPython's readline only
    accepts '\r' as Enter, so a bare LF strands 'hello' in the TIA's line buffer.
    """
    try:
        with serial.Serial(port, baudrate=BAUD_RATE, timeout=timeout) as ser:
            ser.dtr = False          # keep the ESP32-C3 out of its ROM downloader
            ser.rts = False
            time.sleep(0.6)
            ser.reset_input_buffer()
            ser.write(b'hello\r\n')
            ser.flush()
            time.sleep(0.4)
            text = _read_until_quiet(ser)
            banner = next((l.strip() for l in text.splitlines()
                           if l.startswith(PRODUCT_NAME)), None)
            if banner is None:
                ser.dtr = True       # a MicroPython board reads deasserted DTR as 'gone'
                ser.rts = True
                time.sleep(0.2)
            return banner
    except (OSError, serial.SerialException):
        return None


def _first_float(text):
    for line in text.splitlines():
        token = line.replace('>>>', '').replace('getPAR()', '').strip()
        if token:
            try:
                return float(token)
            except ValueError:
                continue
    return None


def probe_tia(port, timeout=2.0, attempts=2, verbose=True):
    """True if `port` answers getPAR() with a number - identifies the board and proves
    the one function we need is live, in one call."""
    for attempt in range(1, attempts + 1):
        try:
            with serial.Serial(port, baudrate=BAUD_RATE, timeout=timeout) as ser:
                ser.dtr = True
                ser.rts = True
                ser.reset_input_buffer()
                ser.reset_output_buffer()
                time.sleep(0.8)
                ser.write(b'getPAR()\r\n')
                ser.flush()
                time.sleep(0.5)
                value = _first_float(_read_until_quiet(ser))
                if value is not None:
                    if verbose:
                        print(f'    {port}: getPAR() -> {value}')
                    return True
        except (OSError, serial.SerialException) as exc:
            if verbose:
                print(f'    {port}: {exc}')
            return False
    return False


print('Scanning serial ports...')
ports = serial_ports()
print(f'  {len(ports)} port(s): {", ".join(ports) if ports else "none"}')

MINIPAR_PORTS = []
for port in ports:
    banner = probe_minipar(port)
    if banner:
        MINIPAR_PORTS.append(port)
        print(f'  {port}: {banner}')
print(f'\n{len(MINIPAR_PORTS)} MiniPAR(s): {", ".join(MINIPAR_PORTS) or "none"}')

print('\nScanning for the reference sensor (TIA)...')
PORT_REF = next((p for p in ports if p not in MINIPAR_PORTS and probe_tia(p)), None)
print(f'  PORT_REF: {PORT_REF}')

print('\nLooking for the LR1-B (USB HID, never a COM port)...')
LR1B_PRESENT = False
if LR1B is None:
    print(f'  lr1b unavailable: {LR1B_IMPORT_ERROR}')
else:
    found = LR1B.list_devices()
    for info in found:
        print(f"  serial {info['serial_number']}  {info['manufacturer_string']} "
              f"{info['product_string']}")
    LR1B_PRESENT = bool(found)
    if not found:
        print('  none found (close the ASEQ application if it is running)')

Scanning serial ports...
  4 port(s): COM66, COM67, COM68, COM69
  COM67: MiniPAR,1.1,1.05
  COM68: MiniPAR,1.1,1.05
  COM69: MiniPAR,1.1,1.05

3 MiniPAR(s): COM67, COM68, COM69

Scanning for the reference sensor (TIA)...
    COM66: getPAR() -> 9.463283
  PORT_REF: COM66

Looking for the LR1-B (USB HID, never a COM port)...
  serial ASQ_SPC2949148  ASEQ Instruments Spectrometer


In [4]:
# Override here if the scan missed something, e.g.
# MINIPAR_PORTS = ['COM64', 'COM65']
# PORT_REF      = 'COM5'

assert MINIPAR_PORTS, 'No MiniPAR found - check the hub, or set MINIPAR_PORTS by hand'
if PORT_REF is None:
    print('WARNING: no TIA. Captures will run with tia_par = NaN.')
if not LR1B_PRESENT:
    print('WARNING: no LR1-B. Captures will run without a reference spectrum.')
print(f'{len(MINIPAR_PORTS)} MiniPAR(s) @ {", ".join(MINIPAR_PORTS)}   '
      f'TIA @ {PORT_REF}   LR1-B: {"yes" if LR1B_PRESENT else "no"}')

3 MiniPAR(s) @ COM67, COM68, COM69   TIA @ COM66   LR1-B: yes


## 4 · Serial / HID layer

Every port is opened **once** and held open — reopening re-asserts DTR/RTS, resets the board
and costs a second or more, which is unusable inside a stability loop.

`MiniPar` gains one method over the original: `spec_sat()`, which reads the firmware's
saturation verdict directly instead of guessing from the counts.

In [5]:
MAC_RE = re.compile(r'^(?:[0-9A-Fa-f]{2}[:-]){5}[0-9A-Fa-f]{2}$')


class MiniPar:
    """Persistent line-mode connection to one MiniPAR (ESP32-C3, USB CDC)."""

    def __init__(self, port, baudrate=BAUD_RATE, timeout=5.0):
        self.port = port
        self.ser = serial.Serial(port, baudrate=baudrate, timeout=timeout)
        self.ser.setDTR(False)
        self.ser.setRTS(False)
        time.sleep(0.3)
        self.ser.write(b'\r\n')          # finish any partial line in the firmware buffer
        self.ser.flush()
        self._drain()
        self.banner = self.handshake()
        if not self.banner:
            self.close()
            raise RuntimeError(f'{port} did not answer as a {PRODUCT_NAME}')
        self.label = self.identify()

    def __repr__(self):
        return f'<{self.label} on {self.port}>'

    def close(self):
        try:
            self.ser.close()
        except Exception:
            pass

    def _drain(self, quiet_s=0.3, max_s=4.0):
        deadline, last = time.monotonic() + max_s, time.monotonic()
        while time.monotonic() < deadline:
            if self.ser.read_all():
                last = time.monotonic()
            elif time.monotonic() - last >= quiet_s:
                return
            time.sleep(0.05)

    def handshake(self, attempts=4):
        for _ in range(attempts):
            self._drain()
            self.ser.write(b'hello\n')
            self.ser.flush()
            time.sleep(0.3)
            for line in self.ser.read_all().decode(errors='ignore').splitlines():
                if line.startswith(PRODUCT_NAME):
                    return line.strip()
        return None

    def device_id(self):
        self._drain()
        self.ser.write(MAC_QUERY.encode())
        self.ser.flush()
        frame = self.ser.readline().decode(errors='ignore').strip()
        match = re.search(r'"device_id"\s*:\s*"([^"]+)"', frame)
        if not match:
            raise RuntimeError(f'No device_id in the JSON frame from {self.port}: {frame!r}')
        return match.group(1), frame

    def get_name(self):
        try:
            return self.cmd('get_name')
        except RuntimeError:
            return ''

    def set_name(self, name):
        return self.cmd(f'set_name,{name}')

    def identify(self):
        """MAC from the JSON header, else the NVS name, else the port."""
        self.device_id_raw, self.frame = self.device_id()
        if MAC_RE.match(self.device_id_raw):
            self.mac = self.device_id_raw.upper().replace('-', ':')
            self.id_source = 'mac'
            return f'miniPAR_{self.mac.replace(":", "")}'
        self.mac = None
        name = self.get_name()
        if name and name.lower() not in {'noname', ''} and not name.startswith('error'):
            self.id_source = 'name'
            return f'miniPAR_{name}'
        self.id_source = 'port'
        return f'miniPAR_{self.port}'

    def cmd(self, text, retries=2):
        for _ in range(retries + 1):
            self.ser.reset_input_buffer()
            self.ser.write((text + '\n').encode())
            self.ser.flush()
            reply = self.ser.readline().decode(errors='ignore').strip()
            if reply:
                if reply.startswith('error:'):
                    raise RuntimeError(f'{self.port} rejected {text!r}: {reply}')
                return reply
        raise RuntimeError(f'No reply from {self.port} for {text!r} '
                           f'(timeout {self.ser.timeout} s)')

    def spec_raw(self):
        """`spec_raw` -> (raw ADC counts, model). Format is frozen in the firmware."""
        reply = self.cmd('spec_raw')
        parts = [p.strip() for p in reply.split(',') if p.strip()]
        model = ''
        if parts and not (parts[0][0].isdigit() or parts[0][0] == '-'):
            model, parts = parts[0], parts[1:]
        if len(parts) != N_SPEC:
            raise RuntimeError(f'{self.label}: expected {N_SPEC} channels, '
                               f'got {len(parts)}: {reply!r}')
        return np.array([float(p) for p in parts], dtype=float), model

    def spec_sat(self):
        """`spec_sat` -> saturation verdict for a fresh reading.

        Reply: `<sat_mask>,<kind>,full_scale,<n>` e.g. `0x0000,none,full_scale,65535`.
        `kind` is none/analog/digital/both. This is the *only* place analog saturation is
        observable - it is a hardware flag and cannot be inferred from the counts.
        """
        reply = self.cmd('spec_sat')
        parts = [p.strip() for p in reply.split(',')]
        if len(parts) < 4 or parts[2] != 'full_scale':
            raise RuntimeError(
                f'{self.label}: unexpected spec_sat reply {reply!r}. This notebook needs the '
                f'firmware build that reports saturation.')
        kind = parts[1]
        return {'sat_mask': parts[0], 'kind': kind, 'full_scale': int(parts[3]),
                'analog': kind in ('analog', 'both'), 'digital': kind in ('digital', 'both')}

    def status(self):
        """`spec_status` -> {'model':..,'available':..,'atime':..,'astep':..,'gain':..}."""
        out = {}
        for field in self.cmd('spec_status').split(','):
            if '=' not in field:
                continue
            key, value = (s.strip() for s in field.split('=', 1))
            try:
                out[key] = int(value)
            except ValueError:
                out[key] = value
        return out

    def par(self, raw=False):
        return float(self.cmd('par_raw' if raw else 'par'))

    def set_setting(self, name, value):
        if name not in {'gain', 'atime', 'astep'}:
            raise ValueError(f'unknown setting {name!r}')
        return int(self.cmd(f'spec_set_{name},{int(value)}'))

    def apply_settings(self, gain, aint, astep):
        """Push all three and verify the device echoes them back."""
        self.set_setting('gain', gain)
        self.set_setting('atime', aint)
        self.set_setting('astep', astep)
        s = self.status()
        got = (s.get('gain'), s.get('atime'), s.get('astep'))
        if got != (int(gain), int(aint), int(astep)):
            raise RuntimeError(f'{self.label}: settings not applied, asked '
                               f'{(gain, aint, astep)}, device reports {got}')
        return s


class TiaRef:
    """Persistent connection to the reference TIA PAR sensor (a MicroPython REPL)."""

    label = 'TIA'

    def __init__(self, port, baudrate=BAUD_RATE, timeout=2.0):
        self.port = port
        self.ser = serial.Serial(port, baudrate=baudrate, timeout=timeout)
        self.ser.dtr = True
        self.ser.rts = True
        time.sleep(0.8)
        self.ser.reset_input_buffer()

    def close(self):
        try:
            self.ser.close()
        except Exception:
            pass

    def par(self, max_lines=8, attempts=2):
        seen = []
        for _ in range(attempts):
            self.ser.reset_input_buffer()
            self.ser.write(b'getPAR()\r\n')
            self.ser.flush()
            for _ in range(max_lines):
                raw = self.ser.readline()
                if not raw:
                    break
                line = raw.decode(errors='ignore').strip()
                if not line:
                    continue
                seen.append(line)
                token = line.replace('>>>', '').replace('getPAR()', '').strip()
                if token:
                    try:
                        return float(token)
                    except ValueError:
                        continue
        raise RuntimeError(f'No numeric PAR from the TIA on {self.port}; saw {seen!r}')


class Lr1bRef:
    """The LR1-B, wrapped so it looks like the other instruments to the capture loop.

    Only this object touches the HID handle, and the capture loop gives it its own thread,
    so the single handle is never used concurrently.
    """

    label = 'LR1B'

    def __init__(self, target_counts=LR1B_TARGET_COUNTS, full_scale=LR1B_FULL_SCALE):
        self.spec = LR1B.discover(full_scale=full_scale)
        self.target_counts = target_counts
        self.exposure_ms = self.spec.parameters.exposure_time_ms
        self.last = None

    @property
    def serial_no(self):
        return self.spec.serial_no

    def close(self):
        try:
            self.spec.close()
        except Exception:
            pass

    def autogain(self, verbose=True):
        """Match the exposure to the ambient level, once, before a capture."""
        result = autogain(self.spec, target_counts=self.target_counts,
                          start_exposure_ms=self.exposure_ms,
                          exposure_min_ms=LR1B_EXPOSURE_MIN,
                          exposure_max_ms=LR1B_EXPOSURE_MAX, verbose=verbose)
        self.exposure_ms = result.exposure_ms
        self.last = result.spectrum
        return result

    def read(self):
        """One spectrum at the fixed, already-auto-gained exposure.

        `discard_first=False`: samples here are back to back, so there is no idle period
        for charge to accumulate over - the flush only matters for the first read, which
        the stability window discards anyway.
        """
        spectrum = self.spec.read_spectrum(self.exposure_ms, discard_first=False)
        self.last = spectrum
        return spectrum


DEVICES, TIA, LR1BREF = [], None, None


def close_links():
    global DEVICES, TIA, LR1BREF
    for link in list(DEVICES) + [x for x in (TIA, LR1BREF) if x is not None]:
        link.close()
    DEVICES, TIA, LR1BREF = [], None, None


def open_links():
    """(Re)open every instrument. Safe to re-run - existing handles are closed first."""
    global DEVICES, TIA, LR1BREF
    close_links()
    opened = []
    for port in MINIPAR_PORTS:
        dev = MiniPar(port)
        opened.append(dev)
        print(f'  {dev.port:8s} {dev.banner:22s} -> {dev.label}  (via {dev.id_source})')

    labels = [d.label for d in opened]
    dupes = {l for l in labels if labels.count(l) > 1}
    for dev in opened:
        if dev.label in dupes:
            dev.label = f'{dev.label}_{dev.port}'
    if dupes:
        print(f'\nWARNING: {len(dupes)} label(s) were not unique; the port has been appended.')

    DEVICES = sorted(opened, key=lambda d: d.label)
    if PORT_REF is not None:
        TIA = TiaRef(PORT_REF)
    if LR1B_PRESENT:
        LR1BREF = Lr1bRef()
        print(f'  LR1-B    serial {LR1BREF.serial_no}  '
              f'{LR1BREF.spec.calibration or "no calibration"}')
    return DEVICES, TIA, LR1BREF


print('Opening instruments...')
open_links()

Opening instruments...
  COM67    MiniPAR,1.1,1.05       -> miniPAR_3CDC750C0518  (via mac)
  COM68    MiniPAR,1.1,1.05       -> miniPAR_3CDC750C0524  (via mac)
  COM69    MiniPAR,1.1,1.05       -> miniPAR_3CDC750C04F4  (via mac)


Expected 3 numeric blocks (3653/3654/3654), found [3653, 7308] in 10975 lines -- the flash read may be incomplete.


  LR1-B    serial ASQ_SPC2949148  LR2B4.0 c.N 3163 | 296.9-1017.5 nm | irr_scaler=1 | irradiance cal: no (blocks found: [3653, 7308])


([<miniPAR_3CDC750C04F4 on COM69>,
  <miniPAR_3CDC750C0518 on COM67>,
  <miniPAR_3CDC750C0524 on COM68>],
 <__main__.Lr1bRef at 0x1fdaf5817f0>)

## 5 · Firmware check

Confirms every MiniPAR answers `spec_sat`, and cross-checks the host-side `full_scale`
formula against the value the firmware computes. A mismatch means the two have drifted apart
and the headroom test would be measuring the wrong thing.

In [6]:
rows, missing, mismatched = [], [], []
for dev in DEVICES:
    s = dev.status()
    try:
        sat = dev.spec_sat()
    except RuntimeError as exc:
        missing.append((dev.label, str(exc)))
        continue
    expected = full_scale_counts(s['atime'], s['astep'])
    if expected != sat['full_scale']:
        mismatched.append((dev.label, expected, sat['full_scale']))
    rows.append({'miniPAR': dev.label, 'port': dev.port, 'model': s.get('model'),
                 'gain': s['gain'], 'aint': s['atime'], 'astep': s['astep'],
                 'tint_ms': round(integration_time_s(s['atime'], s['astep']) * 1e3, 2),
                 'full_scale': sat['full_scale'], 'host_full_scale': expected,
                 'sat_kind': sat['kind'], 'sat_mask': sat['sat_mask']})

display(pd.DataFrame(rows))

if missing:
    for label, exc in missing:
        print(f'{label}: {exc}')
    raise RuntimeError('Flash the saturation-reporting firmware before using this notebook.')
if mismatched:
    for label, expected, got in mismatched:
        print(f'{label}: host full_scale {expected} != firmware {got}')
    raise RuntimeError('full_scale formula disagrees with the firmware.')
print('All devices report saturation, and full_scale agrees with the firmware.')
print('TIA PAR :', TIA.par() if TIA is not None else 'not connected')

,miniPAR,port,model,gain,aint,astep,tint_ms,full_scale,host_full_scale,sat_kind,sat_mask
0,miniPAR_3CDC750C04F4,COM69,AS7341,2,100,999,280.78,65535,65535,none,0x0000
1,miniPAR_3CDC750C0518,COM67,AS7341,2,100,999,280.78,65535,65535,none,0x0000
2,miniPAR_3CDC750C0524,COM68,AS7341,2,100,999,280.78,65535,65535,none,0x0000


All devices report saturation, and full_scale agrees with the firmware.
TIA PAR : 9.485862


## 6 · Randomised settings, validated against both saturation mechanisms

For each device: draw `gain`, then an integration time log-uniformly from
`INT_TIME_MS_RANGE`, then an `astep` and the `aint` that gets closest to it. Apply, read, and
accept only if **all** of these hold at the ambient light level:

* `kind == 'none'` — neither hardware flag is set,
* the **predicted** peak lands inside `PEAK_BAND` (5–70% of full scale) — see below,
* `peak > MIN_PEAK_COUNTS` (200) — a setting that lands the peak at a handful of counts is
  noise, not a measurement,
* `peak <= SAT_HEADROOM * full_scale` — digital headroom, against the *exposure-dependent*
  full scale,
* the analog probe at `ANALOG_MARGIN x` exposure still comes back clean.

A rejection is not just a redraw: it sets a **ceiling** on whichever quantity caused it —
`gain × ASTEP` for analog, `gain × total time` for digital — so a bright room converges in a
few tries instead of rejecting at random forever. Saturation is monotonic in those loads, so
a failure proves only that that load *and above* is unusable; the ceiling excludes exactly
that and nothing more.

> **Why exposure buys less headroom than you would expect.** `full_scale` is
> `(aint+1)·(astep+1)`, so below the 65535 cap the signal *and* the ceiling grow together and
> `peak / full_scale` depends only on **irradiance × gain** — lengthening or shortening the
> integration barely moves it. Exposure only starts to matter once `(aint+1)·(astep+1)`
> exceeds 65535 and the counter clips first. So if a device keeps reporting digital
> saturation, the lever is `gain` (then an ND filter), not the integration time. The
> exhaustion message says which constraint was actually binding.
>
> The **signal floor** is the mirror image: the absolute peak goes as `gain × tint`, so a
> too-dim rejection is answered by raising either. Because the peak is linear in that product,
> the measured shortfall gives the required increase directly — the search jumps straight to
> the right region instead of redrawing blind.

### Responsivity-guided draws

A floor alone is not enough. It admits any setting above 200 counts, so a low-gain draw can be
accepted while sitting at well under 1% of full scale — technically valid, but the noisiest
row in the dataset.

So `measure_responsivity()` first learns one number per device — **counts per gain-second** at
the present light — by climbing from a deliberately dim probe. After that the peak for *any*
`(gain, aint, astep)` is just `responsivity × gain × tint`, and draws are filtered to those
predicted to land inside `PEAK_BAND`. The draw stays random and still spans gain and
integration time; it simply stops proposing settings already known to be useless, so it
usually lands first try. Every clean reading refines the estimate, and the hardware keeps the
final say — responsivity cannot predict *analog* saturation, so `spec_sat` and the margin
probe still run.

Set `RESPONSIVITY_GUIDED = False` for the old floor-only behaviour.

In [7]:
def analog_load(gain, astep):
    """gain x per-step integration time - what drives the analog front end toward railing."""
    return gain_multiplier(gain) * (int(astep) + 1) * AS7341_TICK_S


def digital_load(gain, aint, astep):
    """The quantity `peak / full_scale` actually tracks, monotonically.

    Both the signal and the ceiling grow with exposure - counts go as gain x tint while
    full_scale is tint/TICK - so below the 65535 cap the ratio reduces to `gain x TICK`,
    i.e. it depends on **gain alone**. Only once (aint+1)(astep+1) exceeds 65535 does
    full_scale stop growing and exposure start eating the headroom.

    Using plain gain x tint here would be wrong: it would let the search answer a digital
    rejection by shortening the integration, which in the sub-cap regime changes nothing,
    and never force the gain down.
    """
    return gain_multiplier(gain) * max(AS7341_TICK_S,
                                       integration_time_s(aint, astep) / 0xFFFF)


def signal_load(gain, aint, astep):
    """gain x total integration time - what sets the *absolute* peak count.

    Distinct from digital_load: raising this lifts the peak off the noise floor without
    changing peak/full_scale (below the 65535 cap the ceiling grows with it). So integration
    time is the lever for signal, and gain is the lever for headroom.
    """
    return gain_multiplier(gain) * integration_time_s(aint, astep)


def probe_settings_for(target_load, astep=PROBE_ASTEP):
    """Concrete (gain, aint, astep) delivering `target_load` gain-seconds.

    Walks the gain registers upward so the lowest gain that keeps ATIME in range wins -
    long integration at low gain is the least likely to disturb the analog path, which is
    what we want while only trying to learn the device's response.
    """
    for gain in GAIN_CHOICES:
        aint = aint_for(target_load / gain_multiplier(gain), astep)
        if aint is not None:
            return gain, aint, astep
    return None


def measure_responsivity(dev, tries=5, verbose=False):
    """Counts per gain-second at the present light level.

    With this one number the peak for *any* (gain, aint, astep) is just
    `responsivity x gain x tint`, so settings can be placed analytically instead of by
    trial and error. Starts deliberately dim and climbs, using the measured shortfall to
    size each step, so it neither saturates nor stalls at a couple of counts.
    """
    load = gain_multiplier(0) * 0.1        # 0.5x for 100 ms
    best = None
    for _ in range(tries):
        chosen = probe_settings_for(load)
        if chosen is None:
            break
        gain, aint, astep = chosen
        dev.apply_settings(gain, aint, astep)
        counts, _model = dev.spec_raw()
        sat = dev.spec_sat()
        peak = float(counts.max())
        full_scale = sat['full_scale']
        if verbose:
            print(f'      probe: gain={gain} aint={aint} tint='
                  f'{integration_time_s(aint, astep) * 1e3:6.1f} ms -> peak {peak:6.0f}'
                  f'/{full_scale} {sat["kind"]}')
        if sat['analog'] or sat['digital'] or peak >= SAT_HEADROOM * full_scale:
            load /= 8.0                     # over-range: back off hard and retry
            continue
        best = peak / signal_load(gain, aint, astep)
        if peak >= PROBE_MIN_COUNTS:
            return best
        load *= min(64.0, PROBE_MIN_COUNTS / max(peak, 0.5) * 1.5)
    if best is None:
        raise RuntimeError(f'{dev.label}: could not measure responsivity - the light may be '
                           f'too bright even at {gain_multiplier(0)}x, or the sensor is dark.')
    return best                             # best effort; the band check tolerates error


def draw_settings(digital_ceiling=None, analog_ceiling=None, signal_floor=None,
                  responsivity=None, rng=RNG):
    """One random (gain, aint, astep) respecting both ceilings.

    The two are tracked separately because they are controlled by different registers: the
    counter fills over the *total* time (ATIME x ASTEP), while the integrator rails over a
    *single* step (ASTEP alone). A ceiling on total exposure does not bound ASTEP, so an
    analog rejection has to tighten its own limit or the search can keep drawing a longer
    ASTEP with a smaller ATIME and never converge.
    """
    for _ in range(400):
        gain = rng.choice(GAIN_CHOICES)
        lo, hi = INT_TIME_MS_RANGE
        t_ms = math.exp(rng.uniform(math.log(lo), math.log(hi)))
        astep = rng.randint(*ASTEP_RANGE)
        if analog_ceiling is not None and analog_load(gain, astep) >= analog_ceiling:
            continue
        aint = aint_for(t_ms * 1e-3, astep)
        if aint is None:
            continue
        if digital_ceiling is not None and digital_load(gain, aint, astep) >= digital_ceiling:
            continue
        if signal_floor is not None and signal_load(gain, aint, astep) < signal_floor:
            continue
        if responsivity is not None:
            # Predicted peak must land inside the band: above it the row is SNR-poor
            # (the 200-count floor alone lets a low-gain draw sit at <1 % of range),
            # below the saturation guard it is safe.
            fs = full_scale_counts(aint, astep)
            predicted = responsivity * signal_load(gain, aint, astep)
            lo = max(MIN_PEAK_COUNTS, PEAK_BAND[0] * fs)
            hi = min(SAT_HEADROOM, PEAK_BAND[1]) * fs
            if not (lo <= predicted <= hi):
                continue
        return gain, aint, astep
    raise RuntimeError(
        'Could not draw a setting satisfying the ceilings and the signal floor with '
        f'INT_TIME_MS_RANGE={INT_TIME_MS_RANGE} / ASTEP_RANGE={ASTEP_RANGE} / '
        f'GAIN_CHOICES={GAIN_CHOICES[0]}..{GAIN_CHOICES[-1]}.')


def evaluate_settings(dev, gain, aint, astep, analog_margin=ANALOG_MARGIN):
    """Apply a candidate and report whether it sits far enough from both saturations."""
    dev.apply_settings(gain, aint, astep)
    counts, _model = dev.spec_raw()
    sat = dev.spec_sat()
    full_scale = sat['full_scale']
    peak = float(counts.max())
    headroom = 1.0 - peak / full_scale if full_scale else 0.0

    # `reject` names the mechanism, so the caller tightens the right ceiling instead of
    # pattern-matching prose.
    floor = max(MIN_PEAK_COUNTS, MIN_SIGNAL_FRAC * full_scale)
    verdict = {'gain': gain, 'aint': aint, 'astep': astep, 'peak': peak, 'floor': floor,
               'full_scale': full_scale, 'headroom': headroom,
               'sat_kind': sat['kind'], 'sat_mask': sat['sat_mask'],
               'tint_ms': integration_time_s(aint, astep) * 1e3,
               'analog_margin_ok': None, 'ok': False, 'reason': '', 'reject': None}

    if sat['analog'] or sat['digital']:
        verdict['reason'] = f"hardware reports {sat['kind']} saturation"
        verdict['reject'] = sat['kind']            # analog | digital | both
        return verdict
    if peak > SAT_HEADROOM * full_scale:
        verdict['reason'] = (f'peak {peak:.0f} is above {SAT_HEADROOM:.0%} of full scale '
                             f'{full_scale}')
        verdict['reject'] = 'digital'
        return verdict
    if peak <= floor:
        verdict['reason'] = f'peak {peak:.0f} is not above the {floor:.0f} count floor'
        verdict['reject'] = 'low'
        return verdict

    # Analog margin: the flag has no magnitude, so distance is measured by stressing the
    # analog path and seeing whether it trips.
    #
    # The probe scales ASTEP, not ATIME. ASTEP is the *per-step* integration time, so it is
    # what the integrator charges over and what makes the front end rail; ATIME only sets how
    # many finished steps are accumulated digitally, so raising it would stress the counter
    # and tell us nothing about analog headroom. A digital trip during the probe is expected
    # and ignored - only the analog flag is read. Restored afterwards either way.
    if analog_margin:
        probe_astep = min(65534, int(round((astep + 1) * analog_margin)) - 1)
        if probe_astep <= astep:
            verdict['analog_margin_ok'] = None      # ASTEP is already at its ceiling
        else:
            try:
                dev.apply_settings(gain, aint, probe_astep)
                probe = dev.spec_sat()
            finally:
                dev.apply_settings(gain, aint, astep)
            verdict['analog_margin_ok'] = not probe['analog']
            if probe['analog']:
                verdict['reason'] = (f'analog saturation appears at {analog_margin:g}x ASTEP '
                                     f'- too close to the edge')
                verdict['reject'] = 'analog'
                return verdict

    verdict['ok'] = True
    return verdict


def _setting_advice(attempts):
    """Turn the rejected attempts into advice that names the actual binding constraint.

    Worth being precise here: full_scale is (aint+1)*(astep+1), so below the 65535 cap the
    signal and the full scale grow together and the ratio depends only on irradiance x gain
    - shortening the integration does *not* buy digital headroom. Once gain is at its 0.5x
    floor the only remaining remedy is attenuating the light.
    """
    if not attempts:
        return 'Widen INT_TIME_MS_RANGE / ASTEP_RANGE.'
    kinds = [a['reject'] for a in attempts if a['reject']]
    min_gain = min(a['gain'] for a in attempts)
    best = max(a['headroom'] for a in attempts)
    dominant = max(set(kinds), key=kinds.count) if kinds else 'unknown'
    peak_seen = max(a['peak'] for a in attempts)
    parts = [f'{len(attempts)} attempt(s), mostly {dominant}; best headroom {best:.1%}; '
             f'best peak {peak_seen:.0f} counts; gain tried {min_gain}..'
             f'{max(a["gain"] for a in attempts)}.']
    if dominant in ('digital', 'both'):
        parts.append('Digital headroom below the 65535 cap is set by irradiance x gain, not '
                     'by exposure' + (' - and gain is already at its floor, so attenuate the '
                                      'source (ND filter).' if min_gain == 0 else
                                      ' - allow lower gains in GAIN_CHOICES.'))
    if dominant == 'low':
        parts.append('Signal is below MIN_PEAK_COUNTS: the lever is gain x integration time, '
                     'so allow higher gains or raise the top of INT_TIME_MS_RANGE.')
    parts.append(f'If nothing could be drawn at all, PEAK_BAND={PEAK_BAND} may be out of '
                 'reach at this light - widen it or set RESPONSIVITY_GUIDED = False.')
    if dominant == 'analog':
        parts.append('Analog headroom is set by gain x ASTEP - lower ASTEP_RANGE, allow '
                     'lower gains, or attenuate the source.')
    return ' '.join(parts)


def choose_settings(dev, tries=MAX_SETTING_TRIES, verbose=True):
    """Redraw until a setting clears both saturation tests, tightening after each failure.

    Each rejection halves the ceiling on the quantity that caused it - `gain x ASTEP` for
    analog, `gain x total time` for digital - so a bright room converges in a few tries.
    """
    digital_ceiling = analog_ceiling = signal_floor = None
    responsivity = None
    if RESPONSIVITY_GUIDED:
        responsivity = measure_responsivity(dev, verbose=verbose)
        if verbose:
            print(f'    responsivity {responsivity:,.0f} counts per gain-second')
    attempts = []
    for attempt in range(1, tries + 1):
        try:
            gain, aint, astep = draw_settings(digital_ceiling, analog_ceiling, signal_floor,
                                              responsivity)
        except RuntimeError as exc:
            raise RuntimeError(f'{dev.label}: {exc} {_setting_advice(attempts)}') from None
        verdict = evaluate_settings(dev, gain, aint, astep)
        verdict['attempt'] = attempt
        attempts.append(verdict)
        if verbose:
            mark = 'OK ' if verdict['ok'] else 'no '
            print(f"    {mark} try {attempt}: gain={gain:<2} aint={aint:<4} astep={astep:<6} "
                  f"tint={verdict['tint_ms']:6.1f} ms  peak={verdict['peak']:6.0f}/"
                  f"{verdict['full_scale']:<5} ({verdict['peak'] / verdict['full_scale']:5.1%}"
                  f" of range)"
                  f"{'' if verdict['ok'] else '  <- ' + verdict['reason']}")
        # Every unclipped reading is a fresh measurement of the response, so fold it back
        # in - the probe's estimate is coarse when the probe itself sat near the noise.
        if responsivity is not None and verdict['sat_kind'] == 'none' and verdict['peak'] > 0:
            responsivity = verdict['peak'] / signal_load(gain, aint, astep)

        if verdict['ok']:
            return verdict, attempts

        # Saturation is monotonic in load, so a failure only proves that load and anything
        # above it is unusable. Excluding exactly that keeps every still-viable setting in
        # play; halving the ceiling would discard the usable band just under the limit and
        # can strand the search with nothing left to draw.
        reject = verdict['reject']
        if reject in ('analog', 'both'):
            over = analog_load(gain, astep)
            analog_ceiling = over if analog_ceiling is None else min(analog_ceiling, over)
        if reject in ('digital', 'both'):
            over = digital_load(gain, aint, astep)
            digital_ceiling = over if digital_ceiling is None else min(digital_ceiling, over)
        if reject == 'low':
            # The peak is linear in signal_load, so the measured shortfall says directly how
            # much more is needed - far quicker than redrawing blind at low light.
            need = (signal_load(gain, aint, astep)
                    * verdict['floor'] / max(verdict['peak'], 0.5) * 1.1)
            signal_floor = need if signal_floor is None else max(signal_floor, need)
    raise RuntimeError(f'{dev.label}: no setting cleared saturation in {tries} tries. '
                       + _setting_advice(attempts))


def randomise_all(devices=None, verbose=True):
    devices = DEVICES if devices is None else devices
    chosen = {}
    for dev in devices:
        if verbose:
            print(f'  {dev.label}:')
        verdict, _ = choose_settings(dev, verbose=verbose)
        chosen[dev.label] = verdict
    return chosen


print('Settings sampler ready.')

Settings sampler ready.


In [8]:
# Draw and validate a setting for every device at the current light level.
chosen = randomise_all() if RANDOMISE_SETTINGS else {}
if chosen:
    display(pd.DataFrame([
        {'miniPAR': label, 'gain': v['gain'], 'gain_x': gain_multiplier(v['gain']),
         'aint': v['aint'], 'astep': v['astep'], 'tint_ms': round(v['tint_ms'], 1),
         'peak': v['peak'], 'full_scale': v['full_scale'],
         'of_range': f"{v['peak'] / v['full_scale']:.1%}",
         'headroom': f"{v['headroom']:.1%}", 'sat_kind': v['sat_kind'],
         'analog_margin_ok': v['analog_margin_ok'], 'tries': v['attempt']}
        for label, v in chosen.items()]))

  miniPAR_3CDC750C04F4:
      probe: gain=0 aint=35 tint= 100.1 ms -> peak      3/36000 none
      probe: gain=3 aint=224 tint= 625.5 ms -> peak    101/65535 none
    responsivity 40 counts per gain-second
    OK  try 1: gain=10 aint=8    astep=10869  tint= 272.0 ms  peak=  5171/65535 ( 7.9% of range)
  miniPAR_3CDC750C0518:
      probe: gain=0 aint=35 tint= 100.1 ms -> peak      2/36000 none
      probe: gain=4 aint=143 tint= 400.3 ms -> peak    139/65535 none
    responsivity 43 counts per gain-second
    OK  try 1: gain=10 aint=4    astep=824    tint=  11.5 ms  peak=   244/4125  ( 5.9% of range)
  miniPAR_3CDC750C0524:
      probe: gain=0 aint=35 tint= 100.1 ms -> peak      2/36000 none
      probe: gain=4 aint=143 tint= 400.3 ms -> peak    141/65535 none
    responsivity 44 counts per gain-second
    OK  try 1: gain=10 aint=1    astep=2322   tint=  12.9 ms  peak=   276/4646  ( 5.9% of range)


,miniPAR,gain,gain_x,aint,astep,tint_ms,peak,full_scale,of_range,headroom,sat_kind,analog_margin_ok,tries
0,miniPAR_3CDC750C04F4,10,512.0,8,10869,272.0,5171.0,65535,7.9%,92.1%,none,True,1
1,miniPAR_3CDC750C0518,10,512.0,4,824,11.5,244.0,4125,5.9%,94.1%,none,True,1
2,miniPAR_3CDC750C0524,10,512.0,1,2322,12.9,276.0,4646,5.9%,94.1%,none,True,1


## 7 · Auto-gain the LR1-B

The LR1-B has its own integration time, tuned the same way but with a continuous search
rather than a random draw. It is set **once per capture** and held fixed for the stability
loop — changing exposure mid-loop would make the stability test meaningless. Counts at
different exposures are compared through `counts/ms`.

In [9]:
if LR1BREF is not None:
    lr1b_result = LR1BREF.autogain(verbose=True)
    print(f'\nLR1-B exposure now {LR1BREF.exposure_ms:g} ms')
else:
    lr1b_result = None
    print('No LR1-B connected.')

   1:      0.10 ms -> peak      103 counts
   2:      1.00 ms -> peak       90 counts
   3:     10.00 ms -> peak      134 counts
   4:    100.00 ms -> peak      776 counts
   5:   1000.00 ms -> peak     7995 counts
   6:   2000.00 ms -> peak    16168 counts
Auto-gain NOT converged in 6 steps: 2000 ms -> peak 16168 counts (target 20000, off by -19.2%). Peak only reaches 16168 counts at the maximum exposure (2000 ms) -- raise exposure_max_ms or use averaging.

LR1-B exposure now 2000 ms


## 8 · Stability-gated read across every instrument

One sample submits **all** instruments to the thread pool before awaiting any result, so a
sample describes one instant of light rather than a sweep. The window must be quiet for the
TIA and every MiniPAR at the same time.

**The LR1-B does not gate the capture** (`LR1B_GATES_STABILITY = False`). It is still read
alongside the others and its spread is still reported — marked `*` in the status line to show
it is advisory — but its own noise no longer holds up a capture everything else is ready for.
Its exposure is often hundreds of ms, so it does still set the *sample period*; if that is too
slow, set `LR1B_READ_EVERY_SAMPLE = False` to read it once after the others settle (faster,
but the spectrum is then no longer simultaneous with the MiniPAR readings).

A relative tolerance alone cannot settle near darkness, where a fraction of a count is an
infinite relative change; the `*_ABS_FLOOR` constants give it an absolute escape hatch.

In [10]:
def _spread(values, abs_floor):
    """(peak-to-peak relative to the window mean, whether it is within the absolute floor)."""
    a = np.asarray(values, dtype=float)
    ptp = float(np.ptp(a))
    mean = float(np.mean(np.abs(a)))
    return (ptp / mean if mean > 1e-12 else np.inf), ptp <= abs_floor


def read_stable_multi(devices=None, tia=None, lr1b_ref=None, window=STABLE_WINDOW,
                      tol=STABLE_TOL, max_wait_s=STABLE_MAX_WAIT, settle_s=SETTLE_S,
                      aggregate='median', verbose=True):
    """Sample every instrument until all of them hold steady simultaneously."""
    devices = DEVICES if devices is None else devices
    tia = TIA if tia is None else tia
    lr1b_ref = LR1BREF if lr1b_ref is None else lr1b_ref
    if not devices:
        raise RuntimeError('No devices open - run open_links() first.')

    settings_before = {d.label: d.status() for d in devices}

    tia_hist = deque(maxlen=window)
    spec_hist = {d.label: deque(maxlen=window) for d in devices}
    lr1b_hist = deque(maxlen=window)
    tia_all, lr1b_all = [], []
    total_all = {d.label: [] for d in devices}

    t0, n, stable = time.monotonic(), 0, False
    rel_tia = rel_lr1b = np.inf
    rel_dev = {d.label: np.inf for d in devices}
    lr1b_last = None

    n_workers = len(devices) + (tia is not None) + (lr1b_ref is not None)
    with ThreadPoolExecutor(max_workers=max(1, n_workers)) as pool:
        while True:
            # Everything is submitted before anything is awaited, so the TIA sample, the
            # spectral integrations and the LR1-B exposure overlap in wall clock instead of
            # running back to back.
            tia_future = pool.submit(tia.par) if tia is not None else None
            lr1b_future = (pool.submit(lr1b_ref.read)
                           if lr1b_ref is not None and LR1B_READ_EVERY_SAMPLE else None)
            futures = {dev.label: pool.submit(dev.spec_raw) for dev in devices}

            tia_value = tia_future.result() if tia_future is not None else float('nan')
            lr1b_spec = lr1b_future.result() if lr1b_future is not None else None
            samples = {label: f.result() for label, f in futures.items()}
            n += 1

            tia_hist.append(tia_value)
            tia_all.append(tia_value)
            if lr1b_spec is not None:
                lr1b_last = lr1b_spec
                total = float(np.sum(lr1b_spec.counts))
                lr1b_hist.append(total)
                lr1b_all.append(total)
            for dev in devices:
                spec, _model = samples[dev.label]
                spec_hist[dev.label].append(spec)
                total_all[dev.label].append(float(spec.sum()))

            if len(tia_hist) == window:
                if tia is None:
                    rel_tia, tia_ok = 0.0, True
                else:
                    rel_tia, flat = _spread(tia_hist, TIA_ABS_FLOOR)
                    tia_ok = rel_tia <= tol or flat
                if len(lr1b_hist) == window:
                    rel_lr1b, flat = _spread(lr1b_hist, LR1B_ABS_FLOOR)
                    # Reported for information, but by default it does not gate: its
                    # exposure is long and its own noise would hold up a capture the other
                    # instruments are already ready for.
                    lr1b_ok = (rel_lr1b <= tol or flat) if LR1B_GATES_STABILITY else True
                else:
                    rel_lr1b, lr1b_ok = np.nan, True
                dev_ok = {}
                for dev in devices:
                    totals = [float(s.sum()) for s in spec_hist[dev.label]]
                    rel, flat = _spread(totals, SPEC_ABS_FLOOR)
                    rel_dev[dev.label] = rel
                    dev_ok[dev.label] = rel <= tol or flat
                stable = tia_ok and lr1b_ok and all(dev_ok.values())

                if verbose:
                    worst = max(rel_dev, key=lambda k: rel_dev[k])
                    pending = [k.split('_')[-1][-4:] for k, ok in dev_ok.items() if not ok]
                    if not tia_ok:
                        pending.append('TIA')
                    if LR1B_GATES_STABILITY and not lr1b_ok:
                        pending.append('LR1B')
                    print(f'  [{n:3d}] {time.monotonic() - t0:5.1f} s   '
                          f'TIA {tia_value:9.3f} ({rel_tia * 100:5.2f} %)   '
                          f'LR1B ({rel_lr1b * 100:5.2f} %{"" if LR1B_GATES_STABILITY else "*"})   '
                          f'worst dev {rel_dev[worst] * 100:5.2f} %   '
                          f'waiting on: {", ".join(pending) if pending else "-"}')
                if stable:
                    if verbose:
                        print(f'  -> all stable after {n} samples / '
                              f'{time.monotonic() - t0:.1f} s')
                    break
            elif verbose:
                print(f'  [{n:3d}] filling window ({len(tia_hist)}/{window})')

            if time.monotonic() - t0 > max_wait_s:
                print(f'  WARNING: not all stable within {tol * 100:.1f} % after '
                      f'{max_wait_s:.0f} s - returning the last window anyway.')
                break
            time.sleep(settle_s)

    if lr1b_ref is not None and not LR1B_READ_EVERY_SAMPLE:
        lr1b_last = lr1b_ref.read()      # one spectrum, taken once everything else settled

    out_devices = {}
    for dev in devices:
        after, before = dev.status(), settings_before[dev.label]
        drifted = [k for k in ('gain', 'atime', 'astep') if before.get(k) != after.get(k)]
        if drifted:
            raise RuntimeError(f'{dev.label}: settings changed mid-capture '
                               f'({", ".join(drifted)}): {before} -> {after}')
        arr = np.vstack(spec_hist[dev.label])
        spec_out = np.median(arr, axis=0) if aggregate == 'median' else arr[-1]
        # Authoritative verdict for the reading actually being saved.
        sat = dev.spec_sat()
        peak = float(spec_out.max())
        out_devices[dev.label] = {
            'port': dev.port, 'mac': dev.mac, 'spec': spec_out,
            'model': after.get('model', ''), 'gain': after.get('gain'),
            'aint': after.get('atime'), 'astep': after.get('astep'),
            'spread': rel_dev[dev.label], 'sat_kind': sat['kind'],
            'sat_mask': sat['sat_mask'], 'full_scale': sat['full_scale'],
            'headroom': 1.0 - peak / sat['full_scale'] if sat['full_scale'] else 0.0,
            'saturated_channels': [name for name, p in zip(SPECTRAL_CHANNELS, arr.max(axis=0))
                                   if p >= sat['full_scale']],
            'total_history': total_all[dev.label],
        }

    tia_out = (float('nan') if tia is None else
               float(np.median(np.asarray(tia_hist, dtype=float))) if aggregate == 'median'
               else float(tia_hist[-1]))

    return {
        'tia_par': tia_out, 'devices': out_devices, 'stable': bool(stable),
        'n_samples': n, 'elapsed_s': round(time.monotonic() - t0, 2),
        'tia_spread': rel_tia, 'tia_history': tia_all,
        'lr1b': lr1b_last, 'lr1b_history': lr1b_all, 'lr1b_spread': rel_lr1b,
        'lr1b_exposure_ms': lr1b_ref.exposure_ms if lr1b_ref is not None else float('nan'),
    }


print('read_stable_multi() ready.')

read_stable_multi() ready.


In [11]:
# Smoke test: one stability-gated read across everything, nothing written to disk.
test_reading = read_stable_multi()
print(f"\nTIA PAR {test_reading['tia_par']:.4f}   stable={test_reading['stable']}   "
      f"{test_reading['n_samples']} samples / {test_reading['elapsed_s']} s")
if test_reading['lr1b'] is not None:
    print(f"LR1-B   {test_reading['lr1b']}")
display(pd.DataFrame([
    {'miniPAR': label, 'total': d['spec'].sum(), 'spread_%': round(d['spread'] * 100, 2),
     'gain': d['gain'], 'aint': d['aint'], 'astep': d['astep'],
     'sat_kind': d['sat_kind'], 'headroom': f"{d['headroom']:.1%}"}
    for label, d in test_reading['devices'].items()]))

  [  1] filling window (1/4)
  [  2] filling window (2/4)
  [  3] filling window (3/4)
  [  4]   8.9 s   TIA     5.670 ( 0.00 %)   LR1B ( 2.80 %*)   worst dev  0.93 %   waiting on: -
  -> all stable after 4 samples / 8.9 s

TIA PAR 5.6701   stable=True   4 samples / 9.66 s
LR1-B   Spectrum(2000 ms, n=1): peak 16409 counts @ 456.5 nm, raw peak 22471, baseline 6062


,miniPAR,total,spread_%,gain,aint,astep,sat_kind,headroom
0,miniPAR_3CDC750C04F4,17641.0,0.15,10,8,10869,none,92.1%
1,miniPAR_3CDC750C0518,758.0,0.40,10,4,824,none,94.1%
2,miniPAR_3CDC750C0524,863.0,0.93,10,1,2322,none,94.1%


## 9 · Storage

One row per MiniPAR, as before, plus the saturation verdict and a pointer to the LR1-B
spectrum. The full 3648-point LR1-B spectrum goes to its own file per capture (ASEQ export
format: ascending `wavelength<TAB>counts`) rather than being flattened into the CSV.

In [12]:
def load_readings(path=CSV_PATH):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame(columns=CSV_COLUMNS)
    df = pd.read_csv(path)
    missing = [c for c in CSV_COLUMNS if c not in df.columns]
    if missing:
        raise RuntimeError(f'{path} is missing columns {missing} - it was written with a '
                           f'different schema. Point CSV_PATH at a new file, or migrate it.')
    return df[CSV_COLUMNS]


def next_measurement_id(path=CSV_PATH):
    """max(measurement) + 1, re-read from the file so ids stay unique across sessions."""
    path = Path(path)
    if not path.exists():
        return 1
    col = pd.to_numeric(pd.read_csv(path, usecols=['measurement'])['measurement'],
                        errors='coerce').dropna()
    return int(col.max()) + 1 if not col.empty else 1


def save_lr1b_spectrum(reading, measurement):
    """Write the capture's LR1-B spectrum; returns the filename stored in the CSV."""
    spectrum = reading.get('lr1b')
    if spectrum is None:
        return ''
    LR1B_DIR.mkdir(parents=True, exist_ok=True)
    name = f'measurement_{int(measurement):05d}.txt'
    spectrum.save_txt(LR1B_DIR / name)
    return name


def reading_to_rows(reading, metadata, measurement, lr1b_file=''):
    """One row per device, sharing timestamp / measurement / tia_par / LR1-B summary."""
    stamp = pd.Timestamp.now().isoformat(timespec='seconds')
    spectrum = reading.get('lr1b')
    shared_lr1b = {
        'lr1b_exposure_ms': reading.get('lr1b_exposure_ms', float('nan')),
        'lr1b_peak': float(spectrum.peak) if spectrum is not None else float('nan'),
        'lr1b_total': float(np.sum(spectrum.counts)) if spectrum is not None else float('nan'),
        'lr1b_peak_nm': float(spectrum.peak_wavelength) if spectrum is not None else float('nan'),
        'lr1b_spectrum_file': lr1b_file,
    }
    rows = []
    for label, dev in reading['devices'].items():
        row = {'timestamp': stamp, 'measurement': int(measurement),
               'tia_par': float(reading['tia_par']), 'miniPAR': label, 'metadata': metadata}
        row.update({name: float(v) for name, v in zip(SPECTRAL_CHANNELS, dev['spec'])})
        row.update({'gain': int(dev['gain']), 'aint': int(dev['aint']),
                    'astep': int(dev['astep']), 'sat_kind': dev['sat_kind'],
                    'sat_mask': dev['sat_mask'], 'full_scale': int(dev['full_scale']),
                    'headroom': round(float(dev['headroom']), 4)})
        row.update(shared_lr1b)
        rows.append(row)
    return rows


def append_rows(rows, path=CSV_PATH):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    frame = pd.DataFrame(rows, columns=CSV_COLUMNS)
    frame.to_csv(path, mode='a', index=False, header=not path.exists())
    return frame


_existing = load_readings()
print(f'{len(_existing)} row(s) in {CSV_PATH}; next measurement id = {next_measurement_id()}')

192 row(s) in c:\Users\LudovicoCaracciolo\.traycer\worktrees\jan-ingenhousz-institute__minipar\traycer-minipar-cosmic-elk\Scripts\data\multi_par_spec_lr1b.csv; next measurement id = 65


## 10 · Plots

In [13]:
SURFACE, INK, INK_MUTED, GRID, BAND = '#ffffff', '#1b1b1b', '#8a8a8a', '#e6e6e6', '#dcdcdc'

# Okabe-Ito subset: fixed order, never cycled by rank. Each slot pairs a hue with its own
# marker, so the slots inside the CVD dE 6-8 band stay separable in greyscale and in print.
DEVICE_COLOURS = ['#0072B2', '#D55E00', '#009E73', '#CC79A7', '#56B4E9', '#E69F00']
DEVICE_MARKERS = ['o', 's', '^', 'D', 'v', 'P']
DEVICE_LINES   = ['-', '--', ':']


def device_style(index):
    slot = index % len(DEVICE_COLOURS)
    return (DEVICE_COLOURS[slot], DEVICE_MARKERS[slot],
            DEVICE_LINES[(index // len(DEVICE_COLOURS)) % len(DEVICE_LINES)])


def _style(ax):
    ax.set_facecolor(SURFACE)
    ax.grid(True, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
    for side in ('left', 'bottom'):
        ax.spines[side].set_color(GRID)
    ax.tick_params(colors=INK_MUTED, labelsize=9)
    ax.xaxis.label.set_color(INK_MUTED)
    ax.yaxis.label.set_color(INK_MUTED)
    ax.title.set_color(INK)


def plot_spectra(reading, ax):
    """MiniPAR spectra, normalised per device so shapes compare across exposures."""
    x = np.arange(1, N_SPEC + 1)
    order = {d.label: i for i, d in enumerate(DEVICES)}
    for label, dev in reading['devices'].items():
        colour, marker, dash = device_style(order.get(label, 0))
        spec = np.asarray(dev['spec'], dtype=float)
        total = spec.sum()
        ax.plot(x, spec / total if total else spec, linewidth=2.2, marker=marker,
                markersize=6, color=colour, linestyle=dash, label=label,
                path_effects=[pe.Stroke(linewidth=4, foreground=SURFACE), pe.Normal()])
    ax.set_xticks(x)
    ax.set_xticklabels(SPECTRAL_CHANNELS, rotation=45, ha='right')
    ax.set_xlabel('channel')
    ax.set_ylabel('fraction of total counts')
    ax.set_title(f"MiniPAR spectra - {len(reading['devices'])} device(s)", fontsize=11)
    ax.legend(frameon=False, fontsize=8, labelcolor=INK, loc='best')
    _style(ax)


def plot_lr1b(reading, ax):
    """The LR1-B spectrum for this capture - a single series, so no legend box."""
    spectrum = reading.get('lr1b')
    if spectrum is None:
        ax.text(0.5, 0.5, 'no LR1-B connected', ha='center', va='center',
                transform=ax.transAxes, color=INK_MUTED)
        ax.set_axis_off()
        return
    srt = spectrum.sorted()
    ax.axvspan(400, 700, color=GRID, alpha=0.6, lw=0, zorder=0)
    ax.plot(srt.wavelengths, srt.counts, linewidth=1.3, color='#0072B2', zorder=2)
    ax.set_xlim(srt.wavelengths.min(), srt.wavelengths.max())
    ax.set_xlabel('wavelength (nm)')
    ax.set_ylabel('counts above baseline')
    ax.set_title(f'LR1-B at {reading["lr1b_exposure_ms"]:g} ms - peak '
                 f'{spectrum.peak:,.0f} @ {spectrum.peak_wavelength:.0f} nm', fontsize=11)
    _style(ax)


def plot_convergence(reading, ax):
    """Every instrument's settling trace as % deviation from its own final value.

    Deviation rather than ratio, so the y-axis is directly comparable to STABLE_TOL and
    matplotlib never falls back to `+1` offset notation when the traces are nearly flat.
    """
    def deviation(seq):
        a = np.asarray(seq, dtype=float)
        return (a / a[-1] - 1.0) * 100 if len(a) and abs(a[-1]) > 1e-12 else a * 0.0

    n = reading['n_samples']
    accepted = min(STABLE_WINDOW, n)
    if accepted > 1:
        ax.axvspan(n - accepted + 1, n, color=BAND, alpha=0.55, zorder=0,
                   label=f'accepted window ({accepted})')
    tol = STABLE_TOL * 100
    ax.axhspan(-tol, tol, color='#0072B2', alpha=0.07, zorder=0,
               label=f'tolerance +/-{tol:g} %')
    ax.axhline(0.0, color=GRID, linewidth=1.0, zorder=1)

    order = {d.label: i for i, d in enumerate(DEVICES)}
    for label, dev in reading['devices'].items():
        colour, marker, dash = device_style(order.get(label, 0))
        series = dev['total_history']
        ax.plot(np.arange(1, len(series) + 1), deviation(series), linewidth=1.7,
                marker=marker, markersize=4, color=colour, linestyle=dash,
                label=label, zorder=2)

    tia = np.asarray(reading['tia_history'], dtype=float)
    if np.isfinite(tia).any():
        ax.plot(np.arange(1, len(tia) + 1), deviation(tia), linewidth=2.4,
                linestyle=(0, (6, 3)), color=INK, label='TIA PAR', zorder=4)
    lr = reading.get('lr1b_history') or []
    if len(lr):
        ax.plot(np.arange(1, len(lr) + 1), deviation(lr), linewidth=2.4,
                linestyle=(0, (2, 2)), color='#7a3fb8', label='LR1-B total', zorder=4)

    ax.set_xlabel('sample')
    ax.set_ylabel('deviation from final value (%)')
    ax.set_xticks(np.arange(1, n + 1))
    ax.set_title(f"Convergence - {'all stable' if reading['stable'] else 'NOT stable'} "
                 f"after {n} samples ({reading['elapsed_s']:.1f} s)", fontsize=11)
    ax.legend(frameon=False, fontsize=7, labelcolor=INK, loc='best', ncol=2)
    _style(ax)


def show_capture(reading):
    fig, axes = plt.subplots(1, 3, figsize=(19, 5.4),
                             gridspec_kw={'width_ratios': [1.1, 1.3, 1.1]})
    plot_spectra(reading, axes[0])
    plot_lr1b(reading, axes[1])
    plot_convergence(reading, axes[2])
    fig.patch.set_facecolor(SURFACE)
    plt.tight_layout()
    plt.show()


print('Plot helpers ready.')

Plot helpers ready.


## 10b · Dark-offset characterisation (AN000633 §2.2)

Run this **only** with the sensors capped and the room dark. It is a standalone acquisition
— it does not feed the capture loop below, it writes its own CSV under `data/dark_offset/`.

The sweep spans gain × integration time over ~5.4 decades, log-spaced, because the later
analysis fits `Basic_dark = I_dark + C/(G·T)` per channel: the intercept is dark current, the
slope is the fixed ADC pedestal. Linear spacing would cluster the points and leave that fit
ill-conditioned.

Two things worth knowing before trusting the result:

- **A light leak is indistinguishable from dark current here.** Both are constant in basic
  counts, so no amount of sweeping separates them — the darkness of the enclosure is an
  assumption this data cannot check. The pre-flight burst only catches gross leaks.
- **Dark current is strongly temperature dependent** (roughly doubles per 8–10 °C), so the
  cell asks for ambient temperature and puts it in the filename. Repeat at 2–3 temperatures
  if the offset is to be used outside the lab.


In [ ]:
# =====================================================================================
# Dark-offset characterisation sweep  (AN000633 section 2.2)
# =====================================================================================
# Acquisition only - no fitting, no plots. The later analysis cell fits, per channel
# and per device,
#
#       Basic_dark = I_dark + C / (G * T)
#
# so this sweep must span G*T geometrically. It does: ~5 decades in 7 primary points.
#
# Two design notes that matter for that fit:
#  * The primary ladder varies gain and integration TOGETHER. SPLIT_CONFIGS then hit a
#    G*T that the ladder already covers using a different gain/time split, so the
#    analysis can TEST the "C is a fixed pedestal" assumption instead of assuming it -
#    if C really is fixed, the split points must land on the same line.
#  * full_scale = min(65535, (aint+1)*(astep+1)) shrinks at short exposure, so the
#    low-G*T end is quantisation-limited. That end is exactly what constrains C, hence
#    N_REPEATS reads everywhere and full_scale recorded per config.
# -------------------------------------------------------------------------------------
import datetime as _dt

# --- tunables ------------------------------------------------------------------------
N_REPEATS         = 100      # reads kept per (device, config)
N_DISCARD         = 2        # reads thrown away after each config change (stale integration)
N_TARGET_CONFIGS  = 7        # primary log-spaced G*T points
GT_MIN_S, GT_MAX_S = 1.4e-3, 3.6e2      # target span of gain x integration time (s)
DARKNESS_CLEAR_MAX = 50.0    # basic counts (ams/ms convention) - above this, warn loudly
DARKNESS_BURST     = 10      # reads for the darkness check
ASTEP_PREF         = 999     # preferred ASTEP when solving for a target integration time
LED_OFF            = True    # drive the actinic LED to 0 mA before measuring
OUT_DIR            = DATA_DIR / 'dark_offset'

# Exact field settings, appended as check points. (gain register, aint, astep)
EXTRA_CONFIGS = [
    (2, 100, 999),           # firmware default: 2x, ~280.8 ms
]
# Two configs at IDENTICAL G*T (11.38688 s) reached with a 16x different gain. If C is
# really a fixed pedestal, Basic_dark depends only on G*T and these must agree. If they
# do not, the model needs a gain- or aint-dependent term and the analysis must say so.
SPLIT_CONFIGS = [
    (5, 255, 999),           # 16x,  711.68 ms
    (9, 3, 3999),            # 256x,  44.48 ms
]

CHANNELS = list(SPECTRAL_CHANNELS)
OUT_DIR.mkdir(parents=True, exist_ok=True)


def _solve_exposure(t_target_s, astep_pref=ASTEP_PREF):
    """(aint, astep) whose integration time is as close to t_target_s as the registers allow."""
    steps = max(1, int(round(t_target_s / AS7341_TICK_S)))       # (aint+1)*(astep+1)
    astep_plus = min(max(int(astep_pref), 1), 65535)
    aint_plus = max(1, min(256, int(round(steps / astep_plus))))
    astep_plus = max(1, min(65535, int(round(steps / aint_plus))))
    return aint_plus - 1, astep_plus - 1


def _build_ladder():
    """Primary configs: N_TARGET_CONFIGS log-spaced G*T points across the gain range."""
    targets = np.geomspace(GT_MIN_S, GT_MAX_S, N_TARGET_CONFIGS)
    gains = np.unique(np.round(np.linspace(0, 10, N_TARGET_CONFIGS)).astype(int))
    if len(gains) < N_TARGET_CONFIGS:                    # pad if rounding collapsed any
        gains = np.round(np.linspace(0, 10, N_TARGET_CONFIGS)).astype(int)
    out = []
    for gt, g in zip(targets, gains):
        aint, astep = _solve_exposure(gt / gain_multiplier(g))
        out.append((int(g), int(aint), int(astep)))
    return out


CONFIGS = _build_ladder() + SPLIT_CONFIGS + EXTRA_CONFIGS
_cfg_rows = []
for i, (g, a, s) in enumerate(CONFIGS):
    t = integration_time_s(a, s)
    _cfg_rows.append({'cfg': i, 'gain_reg': g, 'gain_x': gain_multiplier(g), 'aint': a,
                      'astep': s, 't_int_ms': round(t * 1e3, 3),
                      'G*T_s': round(gain_multiplier(g) * t, 6),
                      'full_scale': full_scale_counts(a, s),
                      'kind': ('ladder' if i < len(CONFIGS) - len(SPLIT_CONFIGS) - len(EXTRA_CONFIGS)
                               else 'split' if i < len(CONFIGS) - len(EXTRA_CONFIGS) else 'field')})
_cfg = pd.DataFrame(_cfg_rows)
print('Sweep plan - %d configs, G*T spans %.1f decades'
      % (len(CONFIGS), np.log10(_cfg['G*T_s'].max() / _cfg['G*T_s'].min())))
display(_cfg)

_read_s = float((_cfg.t_int_ms.sum() / 1e3 + 0.05 * len(CONFIGS)) * (N_REPEATS + N_DISCARD))
print('Estimated run time: %.1f min for %d devices (%d reads total)\n'
      % (_read_s * len(DEVICES) / 60.0, len(DEVICES), len(DEVICES) * len(CONFIGS) * (N_REPEATS + N_DISCARD)))

# --- ambient temperature -------------------------------------------------------------
_t_in = input('Ambient temperature at the sensors, in degC (dark current roughly doubles '
              'per 8-10 degC - repeat this sweep at 2-3 temperatures): ').strip()
try:
    AMBIENT_C = float(_t_in)
except ValueError:
    raise SystemExit('A numeric temperature is required; nothing was measured.')

STAMP = _dt.datetime.now(_dt.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT_CSV = OUT_DIR / ('dark_offset_%.1fC_%s.csv' % (AMBIENT_C, STAMP))
print('Writing to %s\n' % OUT_CSV)

# --- LED off -------------------------------------------------------------------------
if LED_OFF:
    for dev in DEVICES:
        try:
            dev.cmd('set_led,0')
        except Exception as exc:
            print('  %s: could not force the LED off (%s)' % (dev.label, exc))

# --- darkness sanity check -----------------------------------------------------------
_gd, _ad, _sd = 10, 255, 999                     # 512x, 711.7 ms - the most sensitive setting
_dark_t = integration_time_s(_ad, _sd)
print('Darkness check at gain %gx, t_int %.1f ms (%d reads/device):'
      % (gain_multiplier(_gd), _dark_t * 1e3, DARKNESS_BURST))
_clear_i = CHANNELS.index('clear')
_suspect = []
for dev in DEVICES:
    try:
        dev.apply_settings(_gd, _ad, _sd)
        for _ in range(N_DISCARD):
            dev.spec_raw()
        vals = np.array([dev.spec_raw()[0] for _ in range(DARKNESS_BURST)])
        basic = vals / (gain_multiplier(_gd) * _dark_t * 1e3)      # ams convention: t in ms
        c_mean, c_std = basic[:, _clear_i].mean(), basic[:, _clear_i].std(ddof=1)
        sat = dev.spec_sat()
        print('  %-24s clear: mean %9.4f  std %8.4f   (raw mean %7.1f)   sat=%s'
              % (dev.label, c_mean, c_std, vals[:, _clear_i].mean(), sat['kind']))
        if c_mean > DARKNESS_CLEAR_MAX:
            _suspect.append((dev.label, c_mean))
    except Exception as exc:
        print('  %-24s FAILED: %s' % (dev.label, exc))

if _suspect:
    print('\n' + '!' * 78)
    print('!! STRAY LIGHT LIKELY. Clear is above %.1f basic counts on: %s'
          % (DARKNESS_CLEAR_MAX, ', '.join('%s (%.2f)' % t for t in _suspect)))
    print('!! A light leak is INDISTINGUISHABLE from dark current in basic counts - both are')
    print('!! constant - so the sweep cannot detect or correct this. Fix the covering first.')
    print('!' * 78)

_ok = input('\nSensors capped and covered, room dark? Type "yes" to start the sweep: ').strip().lower()
if _ok != 'yes':
    raise SystemExit('Not confirmed - nothing was measured.')

# --- the sweep -----------------------------------------------------------------------
_rows, _fails, _done_reads = [], [], 0
_total_reads = len(DEVICES) * len(CONFIGS) * (N_REPEATS + N_DISCARD)
_t0 = time.monotonic()


def _flush():
    """Append whatever is buffered; keeps partial data on disk if this is interrupted."""
    global _rows
    if not _rows:
        return
    frame = pd.DataFrame(_rows)
    frame.to_csv(OUT_CSV, mode='a', index=False, header=not OUT_CSV.exists())
    _rows = []


try:
    for ci, (g, a, s) in enumerate(CONFIGS):
        t_s = integration_time_s(a, s)
        for dev in DEVICES:
            try:
                dev.apply_settings(g, a, s)
                for _ in range(N_DISCARD):                 # drop stale integrations
                    dev.spec_raw()
                    _done_reads += 1
                sat_kind = dev.spec_sat()['kind']
                stamp = _dt.datetime.now(_dt.timezone.utc).isoformat(timespec='seconds')
                for rep in range(N_REPEATS):
                    counts, _model = dev.spec_raw()
                    _done_reads += 1
                    for ch, raw in zip(CHANNELS, counts):
                        _rows.append({'timestamp': stamp, 'ambient_c': AMBIENT_C,
                                      'device_id': dev.label, 'port': dev.port,
                                      'config_index': ci, 'config_kind': _cfg.kind[ci],
                                      'again': gain_multiplier(g), 'gain_reg': g,
                                      'atime': a, 'astep': s,
                                      'integration_time_ms': t_s * 1e3,
                                      'gt_s': gain_multiplier(g) * t_s,
                                      'full_scale': full_scale_counts(a, s),
                                      'sat_kind': sat_kind,
                                      'repeat_index': rep, 'channel': ch,
                                      'raw_count': float(raw)})
            except Exception as exc:
                _fails.append((dev.label, ci, str(exc)))
                print('  %s cfg %d FAILED: %s' % (dev.label, ci, exc))
                _done_reads += N_REPEATS
            _flush()

            frac = _done_reads / max(1, _total_reads)
            el = time.monotonic() - _t0
            eta = el / frac - el if frac > 0 else float('nan')
            print('\r  cfg %2d/%d  %-24s  %5.1f%%   elapsed %5.1f min   ETA %5.1f min'
                  % (ci + 1, len(CONFIGS), dev.label, 100 * frac, el / 60, eta / 60),
                  end='', flush=True)
except KeyboardInterrupt:
    print('\n\nInterrupted - flushing partial data.')
finally:
    _flush()
    print('\n\nSaved %s' % OUT_CSV)

if _fails:
    print('\n%d (device, config) failures:' % len(_fails))
    for lbl, ci, exc in _fails:
        print('  %-24s cfg %d: %s' % (lbl, ci, exc))

# --- compact summary: pedestal significance, before any fitting ----------------------
_d = pd.read_csv(OUT_CSV)
_d['basic_ams'] = _d.raw_count / (_d.again * _d.integration_time_ms)   # t in ms, ams convention
_lo, _hi = _d.gt_s.min(), _d.gt_s.max()
_ends = _d[_d.gt_s.isin([_lo, _hi])]
_piv = (_ends.groupby(['device_id', 'channel', 'gt_s']).basic_ams.mean()
        .unstack('gt_s').rename(columns={_lo: 'lowest_GT', _hi: 'highest_GT'}))
_piv['difference'] = _piv.lowest_GT - _piv.highest_GT
_piv['ratio'] = _piv.lowest_GT / _piv.highest_GT.replace(0, np.nan)
print('\nMean Basic_Count (ams convention, t_int in ms) at the two ends of the G*T range.')
print('A pedestal shows up as lowest_GT >> highest_GT; if the two columns agree, C is')
print('negligible and the offset is pure dark current.  G*T: %.4g s vs %.4g s\n' % (_lo, _hi))
display(_piv.round(5))
_split = _d[_d.config_kind == 'split']
if not _split.empty and _split.gt_s.nunique() == 1:
    _chk = _split.groupby(['device_id', 'channel', 'again']).basic_ams.mean().unstack('again')
    print('\nModel check - identical G*T (%.5f s), %gx vs %gx gain. Equal columns support the'
          % (_split.gt_s.iloc[0], _split.again.min(), _split.again.max()))
    print('fixed-pedestal model; a systematic difference means C depends on gain or aint.\n')
    display(_chk.round(5))

print('\nams workbook reference offsets for scale, same convention (ams units, t in ms):')
print('  F1 0.00197, F2 0.00725, F3 0.00319, F4 0.00131, F5 0.00147, '
      'F6 0.00186, F7 0.00176, F8 0.00522, Clear 0.00300, NIR 0.00100')


## 10c · Dark-offset analysis — fit `Basic_dark = I_dark + C/(G·T)`

Loads every `dark_offset_*.csv` under `data/dark_offset/` (so several temperatures can sit
side by side) and fits the two-term model per channel, per device, per temperature.

| term | meaning | units |
| --- | --- | --- |
| `I_dark` | dark current — the AN000633 §2.2 offset, **if** `C` is negligible | basic counts (ams) |
| `C` | fixed ADC/readout pedestal | raw ADC counts |

**Weighting.** Each config has 100 reads, so the scatter is measured, not assumed. The
single-read sd is floored at `1/√12` counts — the quantisation sd — because a config whose
100 reads are all the same integer would otherwise have zero variance and infinite weight.
The SEM is then divided by `G·T`, so `var(y) ∝ 1/(G·T)²` and the short exposures are
down-weighted: they carry the leverage on `C` but almost none of the precision.

**Three things it checks rather than assumes:**

- `chi2_red` — near 1 means model and weights agree; well above 1 means a missing term, and
  the confidence intervals are widened by `√chi2_red` accordingly.
- the two split configs at *identical* `G·T` but 16× different gain must agree; `|z| > 3`
  says the pedestal depends on gain or aint separately, not just on the product.
- zero-censoring — counts are unsigned, so a true level below ~0.5 count reads as 0 and
  biases the mean high, which would pull `C` upward.

It ends with `constant_offset_ok` per channel: whether a single number is a valid offset at
your field settings, or whether it has to be applied as `I_dark + C/(G·T)` per reading.


In [ ]:
# =====================================================================================
# Dark-offset analysis: fit  Basic_dark = I_dark + C / (G*T)  per channel, per device
# =====================================================================================
# Basic_dark uses the ams convention (integration time in MILLISECONDS), so I_dark is
# directly comparable to the workbook's offset vector, and C comes out in raw ADC counts
# — it is literally "the pedestal, in counts".
#
# Weighting. Each config has N_REPEATS reads, so the scatter is measured rather than
# assumed. Two corrections make that usable at the short-exposure end:
#   * a quantisation floor of 1/sqrt(12) counts on a single read — without it, a config
#     whose 100 reads are all the same integer gets zero variance and infinite weight,
#     which would let one frozen point dictate the whole fit;
#   * the SEM is then divided by (G*T), so var(y) scales as 1/(G*T)^2. Short exposures
#     are therefore correctly *down*-weighted, which is the point: they carry the
#     leverage on C but almost no precision.
#
# chi2_red is the model-adequacy test. If the two-term model is right and the weights are
# honest it sits near 1. Much above 1 means either the model is missing a term (see the
# split-config check) or the noise is underestimated, so the reported CI is widened by
# sqrt(chi2_red) — the "external error" convention.
# -------------------------------------------------------------------------------------
import glob as _glob

IN_DIR       = DATA_DIR / 'dark_offset'
FILE_GLOB    = 'dark_offset_*.csv'
CI_LEVEL     = 0.95
QUANT_SIGMA  = 1.0 / np.sqrt(12.0)   # uniform quantisation sd of one integer read, counts
ZERO_FRAC_WARN = 0.20                # flag a config if this fraction of reads is exactly 0
FIELD_CONFIG = (2, 100, 999)         # settings the offset will actually be used at

try:
    from scipy import stats as _st
    _tcrit = lambda dof: float(_st.t.ppf(0.5 + CI_LEVEL / 2, dof))
except Exception:
    _tcrit = lambda dof: 1.96

_files = sorted(_glob.glob(str(IN_DIR / FILE_GLOB)))
if not _files:
    raise FileNotFoundError('no %s under %s — run the acquisition cell first' % (FILE_GLOB, IN_DIR))
raw = pd.concat([pd.read_csv(f).assign(source_file=Path(f).name) for f in _files],
                ignore_index=True)
print('loaded %d file(s), %d rows' % (len(_files), len(raw)))
print(raw.groupby(['source_file', 'ambient_c']).agg(
    devices=('device_id', 'nunique'), configs=('config_index', 'nunique'),
    reads=('repeat_index', 'nunique')).to_string())

# --- per (temperature, device, channel, config): mean, scatter, censoring -------------
raw['basic'] = raw.raw_count / (raw.again * raw.integration_time_ms)     # ams convention
KEY = ['ambient_c', 'device_id', 'channel', 'config_index']
agg = raw.groupby(KEY).agg(
    gt_s=('gt_s', 'first'), again=('again', 'first'), atime=('atime', 'first'),
    astep=('astep', 'first'), t_ms=('integration_time_ms', 'first'),
    full_scale=('full_scale', 'first'), config_kind=('config_kind', 'first'),
    n=('raw_count', 'size'), raw_mean=('raw_count', 'mean'), raw_sd=('raw_count', 'std'),
    zero_frac=('raw_count', lambda s: float((s == 0).mean())),
    basic_mean=('basic', 'mean')).reset_index()

# sd -> SEM, with the quantisation floor applied to the single-read sd
agg['raw_sd_eff'] = np.maximum(agg.raw_sd.fillna(0.0), QUANT_SIGMA)
agg['sem_raw'] = agg.raw_sd_eff / np.sqrt(agg.n.clip(lower=1))
agg['sigma_basic'] = agg.sem_raw / (agg.again * agg.t_ms)                # var(y) ~ 1/(G*T)^2
agg['x'] = 1.0 / (agg.again * agg.t_ms)                                  # 1/(G*T), ms^-1

_cens = agg[agg.zero_frac > ZERO_FRAC_WARN]
if not _cens.empty:
    print('\n%d (device, channel, config) groups have >%.0f%% of reads at exactly 0.'
          % (len(_cens), 100 * ZERO_FRAC_WARN))
    print('Counts are unsigned, so a true level below ~0.5 count is censored and the mean is')
    print('biased HIGH. These are kept in the fit but listed here — if they cluster at the')
    print('short-exposure end they will pull C upward.\n')
    display(_cens.groupby(['device_id', 'channel']).agg(
        groups=('config_index', 'size'), worst_zero_frac=('zero_frac', 'max'),
        min_gt=('gt_s', 'min')).sort_values('worst_zero_frac', ascending=False).head(12))


def _wls(x, y, sigma):
    """Weighted least squares y = a + b x. Returns a, b, their sd, chi2_red, dof."""
    w = 1.0 / np.maximum(sigma, 1e-300) ** 2
    X = np.column_stack([np.ones_like(x), x])
    XtW = X.T * w
    cov = np.linalg.pinv(XtW @ X)
    beta = cov @ (XtW @ y)
    dof = max(1, len(x) - 2)
    chi2_red = float(np.sum(w * (y - X @ beta) ** 2) / dof)
    sd_int = np.sqrt(np.diag(cov))                       # weights taken as truth
    return beta[0], beta[1], sd_int, chi2_red, dof


rows = []
for (temp, dev, ch), g in agg.groupby(['ambient_c', 'device_id', 'channel']):
    g = g.sort_values('x')
    if len(g) < 3:
        continue
    a, b, sd_int, chi2_red, dof = _wls(g.x.to_numpy(), g.basic_mean.to_numpy(),
                                       g.sigma_basic.to_numpy())
    scale = max(1.0, np.sqrt(chi2_red))                  # external error if the model misfits
    t = _tcrit(dof)
    rows.append({'ambient_c': temp, 'device_id': dev, 'channel': ch, 'n_cfg': len(g),
                 'I_dark': a, 'I_dark_ci': t * sd_int[0] * scale,
                 'C_counts': b, 'C_ci': t * sd_int[1] * scale,
                 'chi2_red': chi2_red})
fit = pd.DataFrame(rows)
fit['I_dark_snr'] = fit.I_dark / fit.I_dark_ci.replace(0, np.nan)
fit['C_snr'] = fit.C_counts / fit.C_ci.replace(0, np.nan)

_ch_order = {c: i for i, c in enumerate(SPECTRAL_CHANNELS)}
fit = fit.sort_values(['ambient_c', 'device_id', 'channel'],
                      key=lambda s: s.map(_ch_order) if s.name == 'channel' else s)

print('\n' + '=' * 92)
print('FIT  Basic_dark = I_dark + C/(G*T)     CI = %.0f%%, widened by sqrt(chi2_red) where >1'
      % (100 * CI_LEVEL))
print('  I_dark   basic counts (ams units) — this is the AN000633 offset if C is negligible')
print('  C        raw ADC counts — the fixed pedestal')
print('=' * 92)
display(fit.set_index(['ambient_c', 'device_id', 'channel']).round(6))

# --- is a constant offset valid at the settings you actually use? --------------------
_fg, _fa, _fs = FIELD_CONFIG
_field_gt_ms = gain_multiplier(_fg) * integration_time_s(_fa, _fs) * 1e3
fit['pedestal_at_field'] = fit.C_counts / _field_gt_ms
fit['pedestal_pct_of_Idark'] = 100 * fit.pedestal_at_field / fit.I_dark.replace(0, np.nan)

print('\nAt the field setting (gain %gx, aint %d, astep %d -> G*T = %.1f gain·ms):'
      % (gain_multiplier(_fg), _fa, _fs, _field_gt_ms))
print('how much does the pedestal term add on top of dark current?\n')
_verdict = (fit.groupby('channel')
            .agg(I_dark=('I_dark', 'mean'), C_counts=('C_counts', 'mean'),
                 pedestal_at_field=('pedestal_at_field', 'mean'),
                 pedestal_pct=('pedestal_pct_of_Idark', 'mean'),
                 C_significant=('C_snr', lambda s: bool((np.abs(s) > 1).all())))
            .reindex(SPECTRAL_CHANNELS))
_verdict['constant_offset_ok'] = (~_verdict.C_significant) | (_verdict.pedestal_pct.abs() < 10)
display(_verdict.round(6))
print('constant_offset_ok = False on any channel means the offset is setting-dependent and')
print('must be applied as I_dark + C/(G*T) per reading, not as one number.')

# --- split-config check: identical G*T, different gain -------------------------------
_sp = agg[agg.config_kind == 'split']
if _sp.empty or _sp.config_index.nunique() < 2:
    print('\nNo split configs found — the fixed-pedestal assumption is untested.')
else:
    _gt = _sp.gt_s.round(6)
    if _gt.nunique() != 1:
        print('\nSplit configs are not at the same G*T (%s) — comparison skipped.'
              % sorted(_gt.unique()))
    else:
        piv = _sp.pivot_table(index=['ambient_c', 'device_id', 'channel'],
                              columns='again', values=['basic_mean', 'sigma_basic'])
        gains = sorted(_sp.again.unique())
        lo, hi = gains[0], gains[-1]
        chk = pd.DataFrame({
            'basic_%gx' % lo: piv[('basic_mean', lo)],
            'basic_%gx' % hi: piv[('basic_mean', hi)],
            'sigma_%gx' % lo: piv[('sigma_basic', lo)],
            'sigma_%gx' % hi: piv[('sigma_basic', hi)]})
        chk['difference'] = chk['basic_%gx' % lo] - chk['basic_%gx' % hi]
        chk['sigma_diff'] = np.hypot(chk['sigma_%gx' % lo], chk['sigma_%gx' % hi])
        chk['z'] = chk.difference / chk.sigma_diff.replace(0, np.nan)
        chk = chk.drop(columns=['sigma_%gx' % lo, 'sigma_%gx' % hi])
        n_bad = int((chk.z.abs() > 3).sum())
        print('\n' + '=' * 92)
        print('MODEL CHECK — identical G*T (%.5f s), %gx vs %gx gain' % (_sp.gt_s.iloc[0], lo, hi))
        print('Basic_dark depends only on G*T if the model holds, so these must agree.')
        print('|z| > 3 on %d of %d rows.' % (n_bad, len(chk)))
        if n_bad:
            print('=> the fixed-pedestal model is INCOMPLETE: something depends on gain or aint')
            print('   separately, not just on their product. Treat C as an effective value.')
        else:
            print('=> consistent with a fixed pedestal; the two-term model is adequate.')
        print('=' * 92)
        display(chk.reindex(sorted(chk.index, key=lambda k: (k[0], k[1], _ch_order.get(k[2], 99))))
                   .round(6))

# --- diagnostics ---------------------------------------------------------------------
print('\nchi2_red distribution: median %.2f, p90 %.2f  (near 1 = model and weights agree)'
      % (fit.chi2_red.median(), fit.chi2_red.quantile(0.9)))

DEV_COLOR = dict(zip(sorted(agg.device_id.unique()), ['#2a78d6', '#eb6834', '#1baf7a']))
SURF, INKC, MUTEDC, GRIDC = '#fcfcfb', '#1a1a19', '#5c5b55', '#e4e3dd'
_temps = sorted(agg.ambient_c.unique())
for temp in _temps:
    sub = agg[agg.ambient_c == temp]
    fig, axes = plt.subplots(2, 5, figsize=(23, 8), facecolor=SURF)
    fig.subplots_adjust(left=0.05, right=0.99, top=0.86, bottom=0.09, wspace=0.26, hspace=0.42)
    fig.suptitle('Dark offset at %.1f °C — Basic_dark vs 1/(G·T), with the weighted fit\n'
                 'intercept = dark current, slope = fixed ADC pedestal'
                 % temp, fontsize=15, color=INKC, x=0.05, ha='left', y=0.97)
    for ax, ch in zip(axes.ravel(), SPECTRAL_CHANNELS):
        ax.set_facecolor(SURF)
        for dev, gd in sub[sub.channel == ch].groupby('device_id'):
            gd = gd.sort_values('x')
            ax.errorbar(gd.x, gd.basic_mean, yerr=gd.sigma_basic, fmt='o', ms=5,
                        color=DEV_COLOR[dev], ecolor=DEV_COLOR[dev], elinewidth=1,
                        capsize=2, alpha=0.9, label=dev[-6:], zorder=3)
            r = fit[(fit.ambient_c == temp) & (fit.device_id == dev) & (fit.channel == ch)]
            if not r.empty:
                xs = np.geomspace(gd.x.min(), gd.x.max(), 60)
                ax.plot(xs, r.I_dark.iloc[0] + r.C_counts.iloc[0] * xs,
                        color=DEV_COLOR[dev], lw=1.2, alpha=0.7, zorder=2)
        ax.set_xscale('log')
        ax.set_title(ch, fontsize=11, color=INKC, loc='left', pad=6)
        ax.set_xlabel('1/(G·T)  (gain·ms)$^{-1}$', fontsize=9, color=MUTEDC)
        ax.set_ylabel('Basic_dark', fontsize=9, color=MUTEDC)
        ax.grid(True, color=GRIDC, lw=0.8, zorder=0)
        ax.set_axisbelow(True)
        for s in ('top', 'right'):
            ax.spines[s].set_visible(False)
        for s in ('left', 'bottom'):
            ax.spines[s].set_color(GRIDC)
        ax.tick_params(colors=MUTEDC, labelsize=8)
    axes.ravel()[0].legend(frameon=False, fontsize=9)
    plt.show()

# --- ready-to-paste offset vectors ---------------------------------------------------
print('\n' + '=' * 92)
print('OFFSET_BASIC per device (ams units) — paste into spectraCalibration_miniPAR_LR1B.ipynb')
print('Valid as a CONSTANT only where constant_offset_ok is True above.')
print('=' * 92)
for (temp, dev), g in fit.groupby(['ambient_c', 'device_id']):
    vec = g.set_index('channel').I_dark.reindex(SPECTRAL_CHANNELS).to_numpy()
    print("# %s @ %.1f degC" % (dev, temp))
    print("OFFSET_BASIC = np.array([%s])" % ', '.join('%.6g' % v for v in vec))


## 11 · Capture loop

Each round: redraw and validate settings (if `RANDOMISE_SETTINGS`), auto-gain the LR1-B, then
wait for everything to be stable together. Review, then type a metadata tag to save — blank
discards. `q` leaves the loop with the ports still open.

In [14]:
readings_df = load_readings()
print(f'{len(readings_df)} row(s) in {CSV_PATH}, next measurement id = {next_measurement_id()}')
print(f'{len(DEVICES)} MiniPAR(s), TIA {"yes" if TIA else "no"}, '
      f'LR1-B {"yes" if LR1BREF else "no"}')

while True:
    key = input('[Enter] capture   |   q + [Enter] quit : ').strip().lower()
    if key in {'q', 'quit', 'exit', 'esc'}:
        print('Left the capture loop. Ports are still open.')
        break

    try:
        if RANDOMISE_SETTINGS:
            print('Drawing acquisition settings...')
            chosen = randomise_all(verbose=True)
        if LR1BREF is not None:
            print('Auto-gaining the LR1-B...')
            LR1BREF.autogain(verbose=False)
            print(f'  exposure {LR1BREF.exposure_ms:g} ms')
        print('Waiting for every instrument to settle...')
        reading = read_stable_multi(verbose=True)
    except Exception as exc:
        print(f'Capture failed: {exc}')
        continue

    clear_output(wait=True)
    show_capture(reading)

    display(pd.DataFrame([
        {'miniPAR': label, 'total': d['spec'].sum(), 'spread_%': round(d['spread'] * 100, 2),
         'gain': d['gain'], 'aint': d['aint'], 'astep': d['astep'],
         'full_scale': d['full_scale'], 'headroom': f"{d['headroom']:.1%}",
         'sat_kind': d['sat_kind']}
        for label, d in reading['devices'].items()]))

    print(f"TIA PAR   : {reading['tia_par']:.4f}  (spread {reading['tia_spread'] * 100:.2f} %)")
    if reading['lr1b'] is not None:
        print(f"LR1-B     : {reading['lr1b']}")
    print(f"stability : {'OK' if reading['stable'] else 'NOT REACHED'} - "
          f"{reading['n_samples']} samples in {reading['elapsed_s']:.1f} s")
    hot = [l for l, d in reading['devices'].items() if d['sat_kind'] != 'none']
    if hot:
        print(f"SATURATED : {', '.join(hot)} - the light changed after the settings were "
              f"validated; redraw before saving")
    if not reading['stable']:
        print('WARNING   : at least one signal was still drifting when this was taken.')

    metadata = input('metadata tag to save this capture (blank = discard): ').strip()
    if not metadata:
        print('Discarded.')
        continue

    measurement = next_measurement_id()
    lr1b_file = save_lr1b_spectrum(reading, measurement)
    rows = reading_to_rows(reading, metadata, measurement, lr1b_file)
    append_rows(rows)
    readings_df = pd.concat([readings_df, pd.DataFrame(rows, columns=CSV_COLUMNS)],
                            ignore_index=True)
    clear_output(wait=True)
    print(f'Saved measurement {measurement} ({len(rows)} rows) to {CSV_PATH}'
          + (f'\n  LR1-B spectrum -> {LR1B_DIR / lr1b_file}' if lr1b_file else '')
          + f'\n  {len(readings_df)} rows total')

Saved measurement 65 (3 rows) to c:\Users\LudovicoCaracciolo\.traycer\worktrees\jan-ingenhousz-institute__minipar\traycer-minipar-cosmic-elk\Scripts\data\multi_par_spec_lr1b.csv
  LR1-B spectrum -> c:\Users\LudovicoCaracciolo\.traycer\worktrees\jan-ingenhousz-institute__minipar\traycer-minipar-cosmic-elk\Scripts\data\lr1b_spectra\measurement_00065.txt
  195 rows total
Left the capture loop. Ports are still open.


## 12 · Review what was collected

In [17]:
df = load_readings()
if df.empty:
    print(f'No readings yet in {CSV_PATH}')
else:
    print(f'{len(df)} rows / {df["measurement"].nunique()} measurement(s)')
    display(df.tail(10))

    print('\nSettings coverage (the point of randomising):')
    display(df.groupby('miniPAR').agg(
        rows=('measurement', 'size'),
        gains=('gain', 'nunique'), aints=('aint', 'nunique'), asteps=('astep', 'nunique'),
        min_headroom=('headroom', 'min'), mean_headroom=('headroom', 'mean')).round(3))

    bad = df[df['sat_kind'] != 'none']
    if len(bad):
        print(f'\n{len(bad)} row(s) saved with saturation:')
        display(bad[['measurement', 'miniPAR', 'sat_kind', 'sat_mask',
                     'full_scale'] + SETTING_COLUMNS])
    else:
        print('\nNo saved row is saturated.')

195 rows / 65 measurement(s)


,timestamp,measurement,tia_par,miniPAR,metadata,f1_415,f2_445,f3_480,f4_515,f5_555,...,astep,sat_kind,sat_mask,full_scale,headroom,lr1b_exposure_ms,lr1b_peak,lr1b_total,lr1b_peak_nm,lr1b_spectrum_file
185,2026-08-11T14:13:57,62,146.626640,miniPAR_3CDC750C0524,fluorcam act1 0 act2 20 fr 0,236.0,1408.0,985.5,2051.0,3184.0,...,5137,none,0x0000,65535,0.9025,64.79,20300.5,7977535.0,452.536,measurement_00062.txt
186,2026-08-11T14:14:26,63,147.891000,miniPAR_3CDC750C04F4,fluorcam act1 0 act2 20 fr 100,186.0,900.0,664.0,1229.0,1998.0,...,10531,none,0x0000,10532,0.5770,64.79,20296.5,8646961.0,451.967,measurement_00063.txt
187,2026-08-11T14:14:26,63,147.891000,miniPAR_3CDC750C0518,fluorcam act1 0 act2 20 fr 100,17.0,83.0,58.0,115.0,179.0,...,3930,none,0x0000,3931,0.8855,64.79,20296.5,8646961.0,451.967,measurement_00063.txt
188,2026-08-11T14:14:26,63,147.891000,miniPAR_3CDC750C0524,fluorcam act1 0 act2 20 fr 100,62.0,293.0,209.0,417.0,647.0,...,14132,none,0x0000,28266,0.9418,64.79,20296.5,8646961.0,451.967,measurement_00063.txt
189,2026-08-11T14:14:55,64,242.945760,miniPAR_3CDC750C04F4,fluorcam act1 20 act2 20 fr 20,16.0,61.0,47.0,85.0,135.0,...,5518,none,0x0000,5519,0.9275,28.33,20277.5,5092500.0,620.859,measurement_00064.txt
190,2026-08-11T14:14:55,64,242.945760,miniPAR_3CDC750C0518,fluorcam act1 20 act2 20 fr 20,184.0,736.0,547.0,1035.0,1595.0,...,8610,none,0x0000,17222,0.6887,28.33,20277.5,5092500.0,620.859,measurement_00064.txt
191,2026-08-11T14:14:55,64,242.945760,miniPAR_3CDC750C0524,fluorcam act1 20 act2 20 fr 20,264.0,1037.0,780.0,1505.0,2286.0,...,3088,none,0x0000,24712,0.6862,28.33,20277.5,5092500.0,620.859,measurement_00064.txt
192,2026-08-11T14:24:15,65,5.692702,miniPAR_3CDC750C04F4,office,42.0,165.0,232.0,264.5,393.5,...,19663,none,0x0000,19664,0.9471,2000.00,16117.5,9067709.0,456.520,measurement_00065.txt
193,2026-08-11T14:24:15,65,5.692702,miniPAR_3CDC750C0518,office,46.0,183.0,247.0,291.5,423.5,...,4294,none,0x0000,21475,0.9406,2000.00,16117.5,9067709.0,456.520,measurement_00065.txt
194,2026-08-11T14:24:15,65,5.692702,miniPAR_3CDC750C0524,office,172.0,682.0,928.5,1125.5,1566.0,...,1825,none,0x0000,65535,0.9274,2000.00,16117.5,9067709.0,456.520,measurement_00065.txt



Settings coverage (the point of randomising):


,rows,gains,aints,asteps,min_headroom,mean_headroom
miniPAR,,,,,,
miniPAR_3CDC750C04F4,65,11,20,65,0.349,0.791
miniPAR_3CDC750C0518,65,10,19,64,0.360,0.796
miniPAR_3CDC750C0524,65,10,23,65,0.374,0.743



No saved row is saturated.


## 13 · Release the instruments

In [18]:
close_links()
print('Serial ports closed and the LR1-B released. Re-run open_links() to reconnect.')

Serial ports closed and the LR1-B released. Re-run open_links() to reconnect.
